# Importing All Libraries

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import StrMethodFormatter
import sklearn
from sklearn.model_selection import train_test_split
import plotly.express as px
import plotly.figure_factory as ff
from dash import dcc, html, Input, Output, Dash
from sklearn.neural_network import MLPClassifier

# Utility Functions

In [0]:
''' 
Description : This function load files from catalog into a dataframe
Input : File Name
Output : Dataframe
'''
def load_file(file_name, header=0):
    df = pd.read_csv(
        f'/Volumes/olist_sales_data/default/olist_sales_volume/{file_name}',
        header=header
    )
    return df

# Loading all data files

In [0]:
try:
    # Loading Customer Dataset
    olist_customers_dataset_df = load_file("olist_customers_dataset.csv", header=0)
    
    # Loading Orders Dataset
    olist_orders_dataset_df = load_file("olist_orders_dataset.csv", header=0)
    
    # Loading Order Items Dataset
    olist_order_items_dataset_df = load_file("olist_order_items_dataset.csv", header=0)
    
    # Loading Products Dataset
    olist_products_dataset_df = load_file("olist_products_dataset.csv", header=0)
    
    # Loading Product Category Translation Dataset
    olist_product_category_translation_df = load_file("product_category_name_translation.csv", header=0)
    
    # Loading Sellers Dataset
    olist_sellers_dataset_df = load_file("olist_sellers_dataset.csv", header=0)
    
    # Loading Geolocation Dataset
    olist_geolocation_dataset_df = load_file("olist_geolocation_dataset.csv", header=0)
    
    # Loading Payment Dataset
    olist_order_payments_dataset_df = load_file("olist_order_payments_dataset.csv", header=0)
    
    # Loading Review Dataset
    olist_order_reviews_dataset_df = load_file("olist_order_reviews_dataset.csv", header=0)
    
    print("All datasets loaded successfully")
except (FileNotFoundError, PermissionError) as e:
    print(f"Cannot access volume: {e}")
    print("Using datasets from previous session if available")

Cannot access volume: [Errno 2] No such file or directory: '/Volumes/olist_sales_data/default/olist_sales_volume/olist_customers_dataset.csv'
Using datasets from previous session if available


In [0]:
olist_customers_dataset_df.head()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8078495091479897>, line 1
----> 1 olist_customers_dataset_df.head()

NameError: name 'olist_customers_dataset_df' is not defined

## EDA – Dataset Overview & Data Quality

In this section, we perform an initial **Exploratory Data Analysis (EDA)** across all eight Olist datasets to understand their structure, characteristics, and data quality before moving into deeper analysis and modeling.

### Key Areas Covered

- **Dataset Size:** Examine the number of **rows and columns** in each dataset to understand the scale of the available data.

- **Data Types:** Review the data type of each column and categorize variables into **numeric, categorical, and datetime** features where applicable.

- **Missing Values:** Identify datasets and individual columns containing **null/missing values**, quantify the extent of missingness, and determine which variables require further investigation or treatment.

- **Duplicate Records:** Check for duplicate rows to identify potential data-quality issues and ensure that records are not unintentionally repeated.

- **Unique Values:** Analyze the number of unique values in each column to understand **cardinality** and identify potential identifier/key columns.

- **Constant Features:** Identify columns with little or no variation that may not provide meaningful information for analysis or modeling.

- **Data Quality Assessment:** Use the findings from this initial assessment to determine the **cleaning, transformation, and preprocessing steps** required before performing detailed EDA, feature engineering, and machine learning.

> **Objective:** The primary objective of this section is to understand the structure and quality of all datasets first, ensuring that subsequent analysis is based on **clean, reliable, and well-understood data**.

In [0]:
# ============================================================
# DATASETS
# ============================================================

datasets = [
    olist_customers_dataset_df,
    olist_geolocation_dataset_df,
    olist_orders_dataset_df,
    olist_order_items_dataset_df,
    olist_order_payments_dataset_df,
    olist_order_reviews_dataset_df,
    olist_products_dataset_df,
    olist_sellers_dataset_df,
    olist_product_category_translation_df
]

names = [
    'Customers',
    'Geolocation',
    'Orders',
    'Order Items',
    'Order Payments',
    'Order Reviews',
    'Products',
    'Sellers'
    'Product Translataion'
]


# ============================================================
# CREATE DATA QUALITY SUMMARY
# ============================================================

summary = []

for name, df in zip(names, datasets):

    total_cells = df.shape[0] * df.shape[1]
    null_count = df.isnull().sum().sum()
    duplicate_count = df.duplicated().sum()

    summary.append({
        'Dataset': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Missing Values': null_count,
        'Missing %': (null_count / total_cells * 100)
                      if total_cells > 0 else 0,
        'Duplicate Rows': duplicate_count,
        'Duplicate %': (duplicate_count / len(df) * 100)
                       if len(df) > 0 else 0,
        'Numeric Columns': df.select_dtypes(
            include=np.number
        ).shape[1],
        'Categorical Columns': df.select_dtypes(
            include=['object', 'category']
        ).shape[1],
        'Unique Columns': sum(
            df[col].is_unique for col in df.columns
        )
    })

summary_df = pd.DataFrame(summary)


# ============================================================
# CHART 1 — NUMBER OF ROWS
# ============================================================

plt.figure(figsize=(12, 6))

plt.bar(
    summary_df['Dataset'],
    summary_df['Rows']
)

plt.title('Number of Rows by Dataset')
plt.xlabel('Dataset')
plt.ylabel('Number of Rows')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


# ============================================================
# CHART 2 — MISSING VALUES
# ============================================================

plt.figure(figsize=(12, 6))

plt.bar(
    summary_df['Dataset'],
    summary_df['Missing %']
)

plt.title('Missing Values by Dataset')
plt.xlabel('Dataset')
plt.ylabel('Missing Values (%)')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


# ============================================================
# CHART 3 — DUPLICATE ROWS
# ============================================================

plt.figure(figsize=(12, 6))

plt.bar(
    summary_df['Dataset'],
    summary_df['Duplicate Rows']
)

plt.title('Duplicate Rows by Dataset')
plt.xlabel('Dataset')
plt.ylabel('Number of Duplicate Rows')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


# ============================================================
# CHART 4 — COLUMN TYPES
# ============================================================

x = np.arange(len(summary_df))
width = 0.35

plt.figure(figsize=(12, 6))

plt.bar(
    x - width / 2,
    summary_df['Numeric Columns'],
    width,
    label='Numeric'
)

plt.bar(
    x + width / 2,
    summary_df['Categorical Columns'],
    width,
    label='Categorical'
)

plt.title('Column Data Types by Dataset')
plt.xlabel('Dataset')
plt.ylabel('Number of Columns')

plt.xticks(
    x,
    summary_df['Dataset'],
    rotation=45,
    ha='right'
)

plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# FINAL SUMMARY TABLE
# ============================================================

display(summary_df.round(2))

In [0]:
# Collections for each dataset
datasets = [olist_customers_dataset_df, olist_geolocation_dataset_df, olist_orders_dataset_df, olist_order_items_dataset_df, olist_order_payments_dataset_df,olist_order_reviews_dataset_df, olist_products_dataset_df, olist_sellers_dataset_df]
names = ['olist_customer', 'olist_geolocation', 'olist_orders', 'olist_order_items', 'olist_order_payments',
         'olist_order_reviews', 'olist_products', 'olist_sellers']

# Creating a DataFrame with useful information about all datasets
data_info = pd.DataFrame({})
data_info['dataset'] = names
data_info['n_rows'] = [df.shape[0] for df in datasets]
data_info['n_cols'] = [df.shape[1] for df in datasets]
data_info['null_amount'] = [df.isnull().sum().sum() for df in datasets]
data_info['qty_null_columns'] = [len([col for col, null in df.isnull().sum().items() if null > 0]) for df in datasets]
data_info['null_columns'] = [', '.join([col for col, null in df.isnull().sum().items() if null > 0]) for df in datasets]

data_info.style.background_gradient()

# Data Cleaning & Preprocessing

- **Handling Missing & Null Values:** Identify missing values and apply appropriate imputation, removal, or treatment strategies based on the context of each variable.

- **Correcting Data Types:** Convert columns to appropriate data types, including numeric, categorical, and datetime formats, to ensure accurate analysis and processing.

- **Handling Duplicate Records:** Identify and remove duplicate records where necessary to maintain data consistency.

- **Data Validation:** Check for invalid, inconsistent, or unexpected values and correct them where appropriate.

- **Standardizing Values:** Normalize inconsistent formats, naming conventions, categorical values, and other data representations across datasets.

- **Handling Outliers:** Identify potential outliers in relevant numerical variables and determine whether they represent valid observations or data-quality issues.

- **Data Integrity Checks:** Validate key identifiers and relationships across datasets to ensure consistency and maintain referential integrity.

> **Objective:** Transform the raw datasets into **clean, consistent, and analysis-ready data** while preserving meaningful information and ensuring data integrity for downstream EDA, feature engineering, and machine learning.

In [0]:
# Orders: parse date columns (missing dates just mean that stage never happened)
olist_orders_dataset_df["order_approved_at"] = pd.to_datetime(olist_orders_dataset_df["order_approved_at"])
olist_orders_dataset_df["order_delivered_carrier_date"] = pd.to_datetime(olist_orders_dataset_df["order_delivered_carrier_date"])
olist_orders_dataset_df["order_delivered_customer_date"] = pd.to_datetime(olist_orders_dataset_df["order_delivered_customer_date"])

# Reviews: no comment written = empty string, not missing data
olist_order_reviews_dataset_df["review_comment_title"] = olist_order_reviews_dataset_df["review_comment_title"].fillna("")
olist_order_reviews_dataset_df["review_comment_message"] = olist_order_reviews_dataset_df["review_comment_message"].fillna("")

# Products: fill category with "unknown", fill numeric columns with median
olist_products_dataset_df["product_category_name"] = olist_products_dataset_df["product_category_name"].fillna("unknown")

num_cols = ["product_name_lenght", "product_description_lenght", "product_photos_qty",
            "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

for col in num_cols:
    olist_products_dataset_df[col] = olist_products_dataset_df[col].fillna(olist_products_dataset_df[col].median())

olist_orders_dataset_df["order_approved_at"] = pd.to_datetime(olist_orders_dataset_df["order_approved_at"])
olist_orders_dataset_df["order_delivered_carrier_date"] = pd.to_datetime(olist_orders_dataset_df["order_delivered_carrier_date"])
olist_orders_dataset_df["order_delivered_customer_date"] = pd.to_datetime(olist_orders_dataset_df["order_delivered_customer_date"])
 
# ---- OPTION 1: DROP rows that have any null in these 3 columns ----
olist_orders_dataset_df = olist_orders_dataset_df.dropna(
    subset=["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]
)

# customers, geolocation, order_items, order_payments, sellers, category_translation
# already have no nulls - nothing to do there

# check
datasets = {
    "customers": olist_customers_dataset_df,
    "geolocation": olist_geolocation_dataset_df,
    "orders": olist_orders_dataset_df,
    "order_items": olist_order_items_dataset_df,
    "order_payments": olist_order_payments_dataset_df,
    "order_reviews": olist_order_reviews_dataset_df,
    "products": olist_products_dataset_df,
    "sellers": olist_sellers_dataset_df,
    "product_category_translation": olist_product_category_translation_df,
}

for name, df in datasets.items():
    print(name, "-> nulls left:", df.isnull().sum().sum())

In [0]:
# Collections for each dataset
datasets = [olist_customers_dataset_df, olist_geolocation_dataset_df, olist_orders_dataset_df, olist_order_items_dataset_df, olist_order_payments_dataset_df,olist_order_reviews_dataset_df, olist_products_dataset_df, olist_sellers_dataset_df]
names = ['olist_customer', 'olist_geolocation', 'olist_orders', 'olist_order_items', 'olist_order_payments',
         'olist_order_reviews', 'olist_products', 'olist_sellers']

# Creating a DataFrame with useful information about all datasets
data_info = pd.DataFrame({})
data_info['dataset'] = names
data_info['n_rows'] = [df.shape[0] for df in datasets]
data_info['n_cols'] = [df.shape[1] for df in datasets]
data_info['null_amount'] = [df.isnull().sum().sum() for df in datasets]
data_info['qty_null_columns'] = [len([col for col, null in df.isnull().sum().items() if null > 0]) for df in datasets]
data_info['null_columns'] = [', '.join([col for col, null in df.isnull().sum().items() if null > 0]) for df in datasets]

data_info.style.background_gradient()

# Business KPIs - Understandign Business Movement

# KPI 1: How did Olist's sales performance change over time?



In [0]:
# Convert order_purchase_timestamp to datetime and create monthly period
olist_orders_dataset_df["order_purchase_timestamp"] = pd.to_datetime(olist_orders_dataset_df["order_purchase_timestamp"])
olist_orders_dataset_df["monthly"] = olist_orders_dataset_df["order_purchase_timestamp"].dt.to_period("M")

# Merge orders with order items to get price and freight information
orders_items = olist_orders_dataset_df.merge(olist_order_items_dataset_df, on="order_id")
orders_items["total_value"] = orders_items["price"] + orders_items["freight_value"]

# Calculate monthly performance metrics
perform = orders_items.groupby("monthly").agg(
    total_sales_value=("total_value", "sum"),
    total_orders=("order_id", "nunique"),
).reset_index()
perform["aov"] = perform["total_sales_value"] / perform["total_orders"]

monthly_viz=perform.copy()
monthly_viz["monthly"]=monthly_viz["monthly"].astype(str)
monthly_viz["total_sales_val_mm"]=monthly_viz["total_sales_value"]/1_000_000
fig,ax=plt.subplots(figsize=(10,4))

monthly_viz.plot(x="monthly",y="total_sales_val_mm",color="#185fa5",
                 linewidth=2,marker="o",ax=ax,legend=False)
ax.set_title("Monthly Sales Value",fontsize=14,fontweight="bold",pad=12)
ax.set_xlabel("Month-Year")
ax.set_ylabel("Sales Value")
ax.set_xticks(range(len(monthly_viz)))
ax.set_xticklabels(monthly_viz["monthly"],ha="right",rotation=45)
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:.1f}M"))
plt.tight_layout()
fig.savefig("monthly_sales.png",dpi=150,bbox_inches="tight")
plt.show()

## KPI 2: Are sales value changes driven by order volume or order value?


In [0]:
monthly_viz=perform.reset_index()
monthly_viz["monthly"]=monthly_viz["monthly"].astype(str)

monthly_viz["sales_val_index"]=(monthly_viz["total_sales_value"]/monthly_viz["total_sales_value"].max())*100
monthly_viz["order_index"]=(monthly_viz["total_orders"]/monthly_viz["total_orders"].max())*100
monthly_viz["aov_index"]=(monthly_viz["aov"]/monthly_viz["aov"].max())*100

fig,ax=plt.subplots(figsize=(10,4))
ax.plot(monthly_viz["monthly"],monthly_viz["sales_val_index"],label="sales value Index",
        color="#185fa5",linewidth=2.5,marker="o")
ax.plot(monthly_viz["monthly"],monthly_viz["order_index"],label="Order Index",
        color="#22e6dc",linewidth=2,linestyle="--",marker="s")
ax.plot(monthly_viz["monthly"],monthly_viz["aov_index"],label="AOV Index",
        color="#cc2e2e",linewidth=2,linestyle=":",marker="^")

ax.set_title("Growth Index Trend: Sales vs Orders vs AOV (Scaled to Peak = 100%)",
             fontsize=12,fontweight="bold",pad=12)
ax.set_xlabel("Month-Year")
ax.set_ylabel("Percentage of Peak Value (%)")

ax.set_xticks(range(len(monthly_viz)))
ax.set_xticklabels(monthly_viz["monthly"],ha="right",rotation=45)
ax.grid(axis="x",linestyle="--",alpha=0.3)
ax.legend(loc="upper left")
plt.tight_layout()
fig.savefig("sales_val_index.png",dpi=150,bbox_inches="tight")
plt.show()

# KPI 3: Which product categories contribute most to sales value?

In [0]:
product_sales_merge=pd.merge(olist_products_dataset_df,olist_order_items_dataset_df,on="product_id")
product_sales_merge["product_sales_value"]=product_sales_merge["freight_value"]+product_sales_merge["price"]
product_sales=product_sales_merge.groupby("product_category_name")["product_sales_value"].sum().reset_index()
product_sales["product_sales_value"]=product_sales["product_sales_value"].round(2)
product_sales=product_sales.sort_values(by="product_sales_value",ascending=False)

top15_products=product_sales.head(15).copy()
top15_products["product_sales_value"]=top15_products["product_sales_value"]/1_000_000

fig,ax=plt.subplots(figsize=(8,6))

norm=mcolors.Normalize(vmin=top15_products["product_sales_value"].min(),
                       vmax=top15_products["product_sales_value"].max())
cmap=plt.cm.Blues
bar_colors=[cmap(norm(val)) for val in top15_products["product_sales_value"]]

bars=ax.barh(top15_products["product_category_name"],top15_products["product_sales_value"],color=bar_colors,
        edgecolor="black",linewidth=0.6)
for bar in bars:
    width=bar.get_width()
    ax.text(
        width+0.1,
        bar.get_y()+bar.get_height()/2,
        f"R${width:.2f}M",
        ha="left",va="center",fontweight="bold"
    )
    ax.margins(x=0.25)
ax.set_title("Top 15 Product Categories by Sales Value",fontsize=14,fontweight="bold",pad=14)
ax.set_xlabel("Sales Value (R$ Millions)",fontsize=10,fontweight="bold")
ax.set_ylabel("Product Category Names",fontsize=10,fontweight="bold")
ax.grid(axis="x",linestyle="--",alpha=0.3)
plt.tight_layout()
fig.savefig("product_sales_value.png",dpi=150,bbox_inches="tight")
plt.show()

In [0]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import pandas as pd

# 1. Category translation and formatting map
category_map = {
    "beleza_saude": "Beauty & Health",
    "relogios_presentes": "Watches & Gifts",
    "cama_mesa_banho": "Bed, Bath & Table",
    "esporte_lazer": "Sports & Leisure",
    "informatica_acessorios": "Computers & Accessories",
    "moveis_decoracao": "Furniture & Decoration",
    "utilidades_domesticas": "Housewares",
    "cool_stuff": "Cool Stuff",
    "automotivo": "Automotive",
    "ferramentas_jardim": "Garden Tools",
    "brinquedos": "Toys",
    "bebes": "Baby Products",
    "perfumaria": "Perfumes & Fragrances",
    "telefonia": "Telephony & Phones",
    "moveis_escritorio": "Office Furniture",
}

# Data preparation
product_sales_merge = pd.merge(
    olist_products_dataset_df, olist_order_items_dataset_df, on="product_id"
)
product_sales_merge["product_sales_value"] = (
    product_sales_merge["freight_value"] + product_sales_merge["price"]
)
product_sales = (
    product_sales_merge.groupby("product_category_name")["product_sales_value"]
    .sum()
    .reset_index()
)
product_sales["product_sales_value"] = product_sales[
    "product_sales_value"
].round(2)
product_sales = product_sales.sort_values(
    by="product_sales_value", ascending=False
)

# Filter top 15 categories
top15_products = product_sales.head(15).copy()
top15_products["product_sales_value"] = (
    top15_products["product_sales_value"] / 1_000_000
)

# 2. Convert category keys to clean label names
top15_products["product_category_name"] = top15_products[
    "product_category_name"
].map(lambda x: category_map.get(x, x.replace("_", " ").title()))

# 3. Sort ascending so the largest value renders at the TOP of the horizontal bar plot
top15_products = top15_products.sort_values(
    by="product_sales_value", ascending=True
)

# Plotting
fig, ax = plt.subplots(figsize=(8, 6))

norm = mcolors.Normalize(
    vmin=top15_products["product_sales_value"].min(),
    vmax=top15_products["product_sales_value"].max(),
)
cmap = plt.cm.Blues
bar_colors = [
    cmap(norm(val)) for val in top15_products["product_sales_value"]
]

bars = ax.barh(
    top15_products["product_category_name"],
    top15_products["product_sales_value"],
    color=bar_colors,
    edgecolor="black",
    linewidth=0.6,
)

# Add value annotations
for bar in bars:
    width = bar.get_width()
    ax.text(
        width + 0.03,  # Reduced offset slightly for better placement
        bar.get_y() + bar.get_height() / 2,
        f"R${width:.2f}M",
        ha="left",
        va="center",
        fontweight="bold",
    )

ax.set_title(
    "Top 15 Product Categories by Sales Value",
    fontsize=14,
    fontweight="bold",
    pad=14,
)
ax.set_xlabel("Sales Value (R$ Millions)", fontsize=10, fontweight="bold")
ax.set_ylabel("Product Category Names", fontsize=10, fontweight="bold")
ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.margins(x=0.2)  # Moved margin adjustment outside the loop

plt.tight_layout()
fig.savefig("product_sales_value.png", dpi=150, bbox_inches="tight")
plt.show()

# KPI 4 : Does sales performance different across customer states?


In [0]:
datasets = [olist_customers_dataset_df, olist_geolocation_dataset_df, olist_orders_dataset_df, olist_order_items_dataset_df, olist_order_payments_dataset_df,olist_order_reviews_dataset_df, olist_products_dataset_df, olist_sellers_dataset_df]

# Use Case 2

In [0]:
# ============================================================
# 1. COPY THE DATASETS
# ============================================================

orders = olist_orders_dataset_df.copy()
items = olist_order_items_dataset_df.copy()
products = olist_products_dataset_df.copy()
customers = olist_customers_dataset_df.copy()
sellers = olist_sellers_dataset_df.copy()


# ============================================================
# 2. PREPARE THE ORDERS DATASET
# ============================================================

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )


# Keep only delivered orders with a delivery date

delivered_orders = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].notna()) &
    (orders["order_estimated_delivery_date"].notna())
].copy()

print("Delivered orders:", len(delivered_orders))

In [0]:
# ============================================================
# 3. CREATE DELIVERY VARIABLES
# ============================================================

# Total time from purchase to delivery

delivered_orders["delivery_time_days"] = (
    delivered_orders["order_delivered_customer_date"] -
    delivered_orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)


# Actual delivery date minus estimated delivery date
# Positive = late
# Negative = early

delivered_orders["delay_days"] = (
    delivered_orders["order_delivered_customer_date"] -
    delivered_orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)


# Time between purchase and approval

delivered_orders["approval_time_hours"] = (
    delivered_orders["order_approved_at"] -
    delivered_orders["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60)


# Time between approval and delivery to the carrier

delivered_orders["seller_processing_days"] = (
    delivered_orders["order_delivered_carrier_date"] -
    delivered_orders["order_approved_at"]
).dt.total_seconds() / (24 * 60 * 60)


# Time between carrier collection and customer delivery

delivered_orders["shipping_time_days"] = (
    delivered_orders["order_delivered_customer_date"] -
    delivered_orders["order_delivered_carrier_date"]
).dt.total_seconds() / (24 * 60 * 60)


# Number of days originally allowed for delivery

delivered_orders["estimated_delivery_days"] = (
    delivered_orders["order_estimated_delivery_date"] -
    delivered_orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)


# Target variable

delivered_orders["late_order"] = (
    delivered_orders["delay_days"] > 0
).astype(int)


# Purchase date variables

delivered_orders["purchase_month"] = (
    delivered_orders["order_purchase_timestamp"].dt.month
)

delivered_orders["purchase_day_of_week"] = (
    delivered_orders["order_purchase_timestamp"].dt.dayofweek
)

delivered_orders["purchase_hour"] = (
    delivered_orders["order_purchase_timestamp"].dt.hour
)

delivered_orders.head()

In [0]:
# ============================================================
# 4. CREATE PRODUCT VARIABLES
# ============================================================

products["product_weight_kg"] = (
    products["product_weight_g"] / 1000
)

products["product_volume_l"] = (
    products["product_length_cm"] *
    products["product_height_cm"] *
    products["product_width_cm"]
) / 1000


# Merge items with product information

items_products = items.merge(
    products[
        [
            "product_id",
            "product_weight_kg",
            "product_volume_l"
        ]
    ],
    on="product_id",
    how="left"
)

print(items_products.shape)
items_products.head()

In [0]:
# Aggregate information so there is one row per order
item_features = items_products.groupby("order_id").agg(
    order_value=("price", "sum"),
    freight_value=("freight_value", "sum"),
    item_count=("order_item_id", "count"),
    unique_products=("product_id", "nunique"),
    unique_sellers=("seller_id", "nunique"),
    average_product_weight_kg=("product_weight_kg", "mean"),
    average_product_volume_l=("product_volume_l", "mean")
).reset_index()


In [0]:
order_customer_state = orders[
    ["order_id", "customer_id"]
].merge(
    customers[
        ["customer_id", "customer_state"]
    ],
    on="customer_id",
    how="left"
)


item_routes = items[
    ["order_id", "seller_id"]
].merge(
    sellers[
        ["seller_id", "seller_state"]
    ],
    on="seller_id",
    how="left"
)


item_routes = item_routes.merge(
    order_customer_state[
        ["order_id", "customer_state"]
    ],
    on="order_id",
    how="left"
)


# 1 if seller and customer are in the same state

item_routes["same_state_item"] = (
    item_routes["seller_state"] ==
    item_routes["customer_state"]
).astype(int)


geography_features = item_routes.groupby("order_id").agg(
    same_state_share=("same_state_item", "mean")
).reset_index()

In [0]:
# ============================================================
# 6. CREATE THE FINAL ANALYSIS DATASET
# ============================================================

analysis_df = delivered_orders.merge(
    customers[
        ["customer_id", "customer_state"]
    ],
    on="customer_id",
    how="left"
)


analysis_df = analysis_df.merge(
    item_features,
    on="order_id",
    how="left"
)


analysis_df = analysis_df.merge(
    geography_features,
    on="order_id",
    how="left"
)


analysis_df["delivery_status"] = "On time or early"

analysis_df.loc[
    analysis_df["late_order"] == 1,
    "delivery_status"
] = "Late"


print("Final dataset size:", analysis_df.shape)

print(
    "Duplicated order IDs:",
    analysis_df["order_id"].duplicated().sum()
)

In [0]:
# ============================================================
# 8. DELAY DISTRIBUTION
# ============================================================

fig = px.histogram(
    analysis_df,
    x="delay_days",
    nbins=80,
    labels={
        "delay_days": "Delay in days"
    },
    title="Distribution of Order Delivery Delays",
    template="simple_white"
)

fig.add_vline(
    x=0,
    line_color="red",
    line_dash="dash"
)

fig.show()

In [0]:
correlation_columns = [
    "delay_days",
    "delivery_time_days",
    "approval_time_hours",
    "seller_processing_days",
    "shipping_time_days",
    "estimated_delivery_days",
    "order_value",
    "freight_value",
    "item_count",
    "unique_products",
    "unique_sellers",
    "average_product_weight_kg",
    "average_product_volume_l",
    "same_state_share"
]


correlation_data = analysis_df[
    correlation_columns
].dropna().copy()


# Remove impossible negative processing times

valid_times = (
    (correlation_data["approval_time_hours"] >= 0) &
    (correlation_data["seller_processing_days"] >= 0) &
    (correlation_data["shipping_time_days"] >= 0) &
    (correlation_data["estimated_delivery_days"] >= 0)
)

correlation_data = correlation_data[valid_times]


correlation_matrix = correlation_data.corr()

delay_correlations = (
    correlation_matrix["delay_days"]
    .drop("delay_days")
    .sort_values()
)

print("\nCORRELATION WITH DELAY")
print("------------------------------------")
print(delay_correlations.round(3))

# Plot correlations

correlation_plot = delay_correlations.reset_index()

correlation_plot.columns = [
    "variable",
    "correlation"
]

fig = px.bar(
    correlation_plot,
    x="correlation",
    y="variable",
    orientation="h",
    color="correlation",
    color_continuous_scale="RdBu_r",
    labels={
        "correlation": "Correlation with delay",
        "variable": "Variable"
    },
    title="Variables Associated with Delivery Delay",
    template="simple_white"
)

fig.show()


In [0]:
# ============================================================
# 10. ESTIMATED DELIVERY TIME VERSUS DELAY
# ============================================================

scatter_data = analysis_df[
    [
        "estimated_delivery_days",
        "delay_days",
        "delivery_status"
    ]
].dropna()

sample_size = min(5000, len(scatter_data))

scatter_data = scatter_data.sample(
    n=sample_size,
    random_state=42
)

fig = px.scatter(
    scatter_data,
    x="estimated_delivery_days",
    y="delay_days",
    color="delivery_status",
    opacity=0.4,
    labels={
        "estimated_delivery_days": "Estimated delivery time in days",
        "delay_days": "Delay in days",
        "delivery_status": "Delivery status"
    },
    title="Estimated Delivery Time versus Delivery Delay",
    template="simple_white"
)

fig.add_hline(
    y=0,
    line_color="red",
    line_dash="dash"
)

fig.show()

In [0]:
# ============================================================
# 7. DELIVERY SUMMARY
# ============================================================

average_delivery_time = analysis_df[
    "delivery_time_days"
].mean()

average_delay = analysis_df[
    "delay_days"
].mean()

median_delay = analysis_df[
    "delay_days"
].median()

late_order_percentage = (
    analysis_df["late_order"].mean() * 100
)

average_late_delay = analysis_df.loc[
    analysis_df["late_order"] == 1,
    "delay_days"
].mean()


print("\nDELIVERY SUMMARY")
print("------------------------------------")
print(f"Number of delivered orders: {len(analysis_df):,}")
print(f"Average delivery time: {average_delivery_time:.2f} days")
print(f"Average delay: {average_delay:.2f} days")
print(f"Median delay: {median_delay:.2f} days")
print(f"Percentage of late orders: {late_order_percentage:.2f}%")
print(f"Average delay among late orders: {average_late_delay:.2f} days")
#display(analysis_df[analysis_df[
#    "delay_days"
#]==188.97508101851852]) data skewed, skewedness check

In [0]:
state_summary = analysis_df.groupby(
    "customer_state"
).agg(
    order_count=("order_id", "count"),
    average_delay_days=("delay_days", "mean"),
    late_order_percentage=("late_order", "mean")
).reset_index()


state_summary["late_order_percentage"] = (
    state_summary["late_order_percentage"] * 100
)


# Keep states with at least 100 orders

state_summary = state_summary[
    state_summary["order_count"] >= 100
].sort_values(
    by="late_order_percentage",
    ascending=False
)


print("\nDELAY BY CUSTOMER STATE")
print("------------------------------------")
print(state_summary.round(2))


fig = px.bar(
    state_summary,
    x="customer_state",
    y="late_order_percentage",
    color="late_order_percentage",
    labels={
        "customer_state": "Customer state",
        "late_order_percentage": "Late orders (%)"
    },
    title="Percentage of Late Orders by Customer State",
    template="simple_white"
)

fig.show()


In [0]:
# ============================================================
# 12. DELAY PREDICTION
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# Only use variables available when the order is placed.
# Do not use actual shipping or delivery times.

prediction_features = [
    "estimated_delivery_days",
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour",
    "order_value",
    "freight_value",
    "item_count",
    "unique_products",
    "unique_sellers",
    "average_product_weight_kg",
    "average_product_volume_l",
    "same_state_share"
]


model_data = analysis_df[
    prediction_features + ["late_order"]
].dropna().copy()


X = model_data[prediction_features]
Y = model_data["late_order"]


print("\nMODEL DATA")
print("------------------------------------")
print("Model observations:", len(model_data))

print("\nTarget percentages:")
print((Y.value_counts(normalize=True) * 100).round(2))


# Split into training and testing data

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.20,
    shuffle=True,
    random_state=42,
    stratify=Y
)


# Scale the variables

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Train the logistic regression

delay_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

delay_model.fit(
    X_train_scaled,
    Y_train
)


# Make predictions

delay_predictions = delay_model.predict(
    X_test_scaled
)

delay_risk_score = delay_model.predict_proba(
    X_test_scaled
)[:, 1]


# ============================================================
# COMPARE DIFFERENT THRESHOLDS
# ============================================================

thresholds = [
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

for threshold in thresholds:

    predictions = (
        delay_risk_score >= threshold
    ).astype(int)

    accuracy = accuracy_score(
        Y_test,
        predictions
    )

    precision = precision_score(
        Y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        Y_test,
        predictions
    )

    print(f"\nThreshold: {threshold}")
    print(f"Accuracy: {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall: {recall:.3f}")



In [0]:
chosen_threshold = 0.5

delay_predictions = (
    delay_risk_score >= chosen_threshold
).astype(int)

accuracy = accuracy_score(
    Y_test,
    delay_predictions
)

precision = precision_score(
    Y_test,
    delay_predictions,
    zero_division=0
)

recall = recall_score(
    Y_test,
    delay_predictions
)

confusion = confusion_matrix(
    Y_test,
    delay_predictions
)

print("\nFINAL MODEL RESULTS")
print("------------------------------------")
print(f"Chosen threshold: {chosen_threshold}")
print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")

print("\nConfusion matrix:")
print(confusion)

# ============================================================
# 14. MODEL COEFFICIENTS
# ============================================================

coefficient_table = pd.DataFrame({
    "feature": prediction_features,
    "coefficient": delay_model.coef_[0]
})

coefficient_table = coefficient_table.sort_values(
    by="coefficient",
    ascending=False
)

print("\nMODEL COEFFICIENTS")
print("------------------------------------")
print(coefficient_table)


# ============================================================
# 15. SAVE TEST PREDICTIONS IN A DATAFRAME
# ============================================================

prediction_results = X_test.copy()

prediction_results["actual_late"] = Y_test
prediction_results["predicted_late"] = delay_predictions
prediction_results["delay_risk_score"] = delay_risk_score

prediction_results = prediction_results.sort_values(
    by="delay_risk_score",
    ascending=False
)

prediction_results.head(20)

In [0]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Random forest does not require scaled data
random_forest_model.fit(
    X_train,
    Y_train
)

rf_predictions = random_forest_model.predict(
    X_test
)

rf_risk_score = random_forest_model.predict_proba(
    X_test
)[:, 1]

In [0]:
rf_accuracy = accuracy_score(
    Y_test,
    rf_predictions
)

rf_precision = precision_score(
    Y_test,
    rf_predictions,
    zero_division=0
)

rf_recall = recall_score(
    Y_test,
    rf_predictions
)

rf_confusion = confusion_matrix(
    Y_test,
    rf_predictions
)

print("RANDOM FOREST RESULTS")
print("------------------------------------")
print(f"Accuracy: {rf_accuracy:.3f}")
print(f"Precision: {rf_precision:.3f}")
print(f"Recall: {rf_recall:.3f}")

print("\nConfusion matrix:")
print(rf_confusion)

In [0]:
model_comparison = pd.DataFrame({
    "model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "accuracy": [
        accuracy,
        rf_accuracy
    ],
    "precision": [
        precision,
        rf_precision
    ],
    "recall": [
        recall,
        rf_recall
    ]
})

print(model_comparison.round(3))

Checking for skewedness and rectifying

In [0]:
#skewness check
feature_skewness = (
    model_data[prediction_features]
    .skew()
    .sort_values(ascending=False)
)

print("FEATURE SKEWNESS")
print("------------------------------------")
print(feature_skewness.round(3))


In [0]:
X_transformed = X.copy()

features_to_log = [
    "freight_value",
    "order_value",
    "average_product_volume_l",
    "average_product_weight_kg",
    "estimated_delivery_days"
]

for feature in features_to_log:
    X_transformed[feature] = np.log1p(
        X_transformed[feature]
    )

In [0]:
X_transformed["multiple_items"] = (
    X_transformed["item_count"] > 1
).astype(int)

X_transformed["multiple_products"] = (
    X_transformed["unique_products"] > 1
).astype(int)

X_transformed["multiple_sellers"] = (
    X_transformed["unique_sellers"] > 1
).astype(int)

X_transformed = X_transformed.drop(
    columns=[
        "item_count",
        "unique_products",
        "unique_sellers"
    ]
)

Transformed logistic model

In [0]:
X_train_log, X_test_log, Y_train_log, Y_test_log = train_test_split(
    X_transformed,
    Y,
    test_size=0.20,
    shuffle=True,
    random_state=42,
    stratify=Y
)

log_scaler = StandardScaler()

X_train_log_scaled = log_scaler.fit_transform(
    X_train_log
)

X_test_log_scaled = log_scaler.transform(
    X_test_log
)

log_delay_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

log_delay_model.fit(
    X_train_log_scaled,
    Y_train_log
)

log_predictions = log_delay_model.predict(
    X_test_log_scaled
)

print("Accuracy:", round(
    accuracy_score(Y_test_log, log_predictions), 3
))

print("Precision:", round(
    precision_score(
        Y_test_log,
        log_predictions,
        zero_division=0
    ), 3
))

print("Recall:", round(
    recall_score(Y_test_log, log_predictions), 3
))

In [0]:
X_train_rf_log, X_test_rf_log, Y_train_rf_log, Y_test_rf_log = (
    train_test_split(
        X_transformed,
        Y,
        test_size=0.20,
        shuffle=True,
        random_state=42,
        stratify=Y
    )
)


# Train Random Forest using transformed variables

transformed_rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

transformed_rf_model.fit(
    X_train_rf_log,
    Y_train_rf_log
)


# Make predictions

transformed_rf_predictions = transformed_rf_model.predict(
    X_test_rf_log
)


# Evaluate the model

transformed_rf_accuracy = accuracy_score(
    Y_test_rf_log,
    transformed_rf_predictions
)

transformed_rf_precision = precision_score(
    Y_test_rf_log,
    transformed_rf_predictions,
    zero_division=0
)

transformed_rf_recall = recall_score(
    Y_test_rf_log,
    transformed_rf_predictions
)


transformed_rf_confusion = confusion_matrix(
    Y_test_rf_log,
    transformed_rf_predictions
)


print("TRANSFORMED RANDOM FOREST")
print("------------------------------------")
print(f"Accuracy: {transformed_rf_accuracy:.3f}")
print(f"Precision: {transformed_rf_precision:.3f}")
print(f"Recall: {transformed_rf_recall:.3f}")

print("\nConfusion matrix:")
print(transformed_rf_confusion)

Neural network

In [0]:
nn_training_data = X_train.copy()

nn_training_data["late_order"] = Y_train


# Separate on-time and late orders

on_time_training = nn_training_data[
    nn_training_data["late_order"] == 0
]

late_training = nn_training_data[
    nn_training_data["late_order"] == 1
]


# Duplicate late orders until both classes have the same size

late_training_balanced = late_training.sample(
    n=len(on_time_training),
    replace=True,
    random_state=42
)


# Combine and shuffle the balanced training data

balanced_training = pd.concat([
    on_time_training,
    late_training_balanced
])

balanced_training = balanced_training.sample(
    frac=1,
    random_state=42
)


X_train_nn = balanced_training[
    prediction_features
]

Y_train_nn = balanced_training[
    "late_order"
]


print("NEURAL NETWORK TRAINING DISTRIBUTION")
print("------------------------------------")
print(Y_train_nn.value_counts())

print("\nPercentages:")
print(
    (
        Y_train_nn.value_counts(normalize=True) * 100
    ).round(2)
)

In [0]:
# ============================================================
# 2. SCALE THE FEATURES
# ============================================================

nn_scaler = StandardScaler()

# Fit the scaler using the original training distribution
nn_scaler.fit(X_train)

X_train_nn_scaled = nn_scaler.transform(
    X_train_nn
)

X_test_nn_scaled = nn_scaler.transform(
    X_test
)

In [0]:
# ============================================================
# 3. TRAIN THE NEURAL NETWORK
# ============================================================

neural_network_model = MLPClassifier(
    hidden_layer_sizes=(20, 20),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=10000,
    validation_fraction=0.10,
    random_state=42
)

neural_network_model.fit(
    X_train_nn_scaled,
    Y_train_nn
)

In [0]:
# ============================================================
# 4. MAKE PREDICTIONS
# ============================================================

nn_predictions = neural_network_model.predict(
    X_test_nn_scaled
)

nn_risk_score = neural_network_model.predict_proba(
    X_test_nn_scaled
)[:, 1]

# ============================================================
# 5. EVALUATE THE NEURAL NETWORK
# ============================================================

nn_accuracy = accuracy_score(
    Y_test,
    nn_predictions
)

nn_precision = precision_score(
    Y_test,
    nn_predictions,
    zero_division=0
)

nn_recall = recall_score(
    Y_test,
    nn_predictions
)


nn_confusion = confusion_matrix(
    Y_test,
    nn_predictions
)


print("NEURAL NETWORK RESULTS")
print("------------------------------------")
print(f"Accuracy: {nn_accuracy:.3f}")
print(f"Precision: {nn_precision:.3f}")
print(f"Recall: {nn_recall:.3f}")

print("\nConfusion matrix:")
print(nn_confusion)

print(
    "\nTraining iterations:",
    neural_network_model.n_iter_
)

Model comparison


In [0]:
def evaluate_model(model_name, y_true, y_prediction):

    return {
        "model": model_name,
        "accuracy": accuracy_score(
            y_true,
            y_prediction
        ),
        "precision": precision_score(
            y_true,
            y_prediction,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            y_prediction
        )
    }


four_model_comparison = pd.DataFrame([
    
    evaluate_model(
        "Original Logistic Regression",
        Y_test,
        delay_predictions
    ),
    
    evaluate_model(
        "Transformed Logistic Regression",
        Y_test_log,
        log_predictions
    ),
    
    evaluate_model(
        "Original Random Forest",
        Y_test,
        rf_predictions
    ),
    
    evaluate_model(
        "Transformed Random Forest",
        Y_test_rf_log,
        transformed_rf_predictions
    ),
    evaluate_model(
        "Original Neural Network",
        Y_test,
        nn_predictions
    )
])

display(
    four_model_comparison.round(3)
)

### Order Review

In [0]:
reviews = olist_order_reviews_dataset_df.copy()

reviews['has_comments']= reviews['review_comment_message']!=""

# How often are customers writing comment review, grouped by review score
comment_rate = (
    reviews.groupby('review_score')['has_comments']
      .mean()*100
)

print(comment_rate)

reviews['comment_length'] = (
    reviews['review_comment_message']
    .dropna()
    .str.len()
)

# Histogram of review comment length
plt.hist(
    reviews.loc[reviews['has_comments'], 'comment_length'],
    bins=30
)

plt.xlabel('Comment Length')
plt.ylabel('Number of Reviews')
plt.title('Distribution of Review Comment Length')

# Boxplot of review comment length by review score
reviews[reviews['has_comments']].boxplot(
    column='comment_length',
    by='review_score'
)

plt.xlabel('Review Score')
plt.ylabel('Comment Length')
plt.title('Comment Length by Review Score')
plt.suptitle('')

plt.show()

# Handling empty comments
reviews_no_empty_comment = reviews[reviews['review_comment_message']!=""]

reviews['comment_length'] = reviews_no_empty_comment['review_comment_message'].str.len()

# Median character length of comments by review score
median_comment_length = (
    reviews_no_empty_comment.groupby('review_score')['comment_length']
      .median()
)

# Median word count of comments by review score
reviews_no_empty_comment['word_count'] = (
    reviews_no_empty_comment['review_comment_message']
    .str.split()
    .str.len()
)

print(reviews_no_empty_comment.groupby('review_score')['word_count'].median())

print(median_comment_length)

# Review date conversions
review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in review_date_columns:
    reviews[column] = pd.to_datetime(
        reviews[column],
        errors="coerce"
    )

reviews['review_month'] = (
    reviews['review_creation_date']
    .dt.to_period('M')
)

monthly_score = (
    reviews.groupby('review_month')['review_score']
      .mean()
)
# Plot average review score over time: Does customer satisfaction increase or decrease over time?
monthly_score.plot()

plt.xlabel('Month')
plt.ylabel('Average Review Score')
plt.title('Average Review Score Over Time')

plt.show()

#
df_orders_reviews = pd.merge(
    reviews,
    orders,
    on='order_id',
    how='inner'
)

df_orders_reviews = df_orders_reviews[
    (df_orders_reviews['order_status'] == 'delivered') &
    (df_orders_reviews['order_delivered_customer_date'].notna())
].copy()

df_orders_reviews['delivery_days'] = (
    df_orders_reviews['order_delivered_customer_date']
    - df_orders_reviews['order_purchase_timestamp']
).dt.days

df_orders_reviews['late'] = (
    df_orders_reviews['order_delivered_customer_date'].dt.normalize()
    >
    df_orders_reviews['order_estimated_delivery_date'].dt.normalize()
)

df_orders_reviews.groupby('late')['review_score'].mean()

df_orders_reviews['delay_days'] = (
    df_orders_reviews['order_delivered_customer_date'].dt.normalize()
    -
    df_orders_reviews['order_estimated_delivery_date'].dt.normalize()
).dt.days

bins = [-np.inf, -14, -7, 0, 3, 7, 14, np.inf]

labels = [
    '≥14 days early',
    '7–14 days early',
    '0–7 days early/on date',
    '1–3 days late',
    '4–7 days late',
    '8–14 days late',
    '>14 days late'
]

df_orders_reviews['delivery_timing'] = pd.cut(
    df_orders_reviews['delay_days'],
    bins=bins,
    labels=labels
)

result = df_orders_reviews.groupby(
    'delivery_timing',
    observed=False
)['review_score'].agg(
    avg_rating='mean',
    one_star_percent=lambda x: (x == 1).mean() * 100
)

# Rounding
result['avg_rating'] = result['avg_rating'].round(2)
result['one_star_percent'] = result['one_star_percent'].round(1)

print(result)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8078495091480022>, line 1
----> 1 reviews = olist_order_reviews_dataset_df.copy()
      3 reviews['has_comments']= reviews['review_comment_message']!=""
      5 # How often are customers writing comment review, grouped by review score

NameError: name 'olist_order_reviews_dataset_df' is not defined

In [0]:
# ============================================================
# PREPARE DATA FOR BUSINESS IMPACT ANALYSIS
# ============================================================
# Add review_score and repeat_customer columns to analysis_df

# Check prerequisites
if 'olist_order_reviews_dataset_df' not in dir():
    raise NameError("❌ Please run Cell 8 first to load olist_order_reviews_dataset_df")
if 'analysis_df' not in dir():
    raise NameError("❌ Please run Cell 33 first to create analysis_df")

# 1. Merge review scores into analysis_df
reviews_data = olist_order_reviews_dataset_df[['order_id', 'review_score']].copy()
analysis_df = analysis_df.merge(reviews_data, on='order_id', how='left')

print(f"✓ Added review_score column to analysis_df")
print(f"  - Orders with reviews: {analysis_df['review_score'].notna().sum():,}")
print(f"  - Average review score: {analysis_df['review_score'].mean():.2f}")

# 2. Create repeat_customer flag (1 if customer has multiple orders, 0 otherwise)
customer_order_counts = analysis_df.groupby('customer_id')['order_id'].transform('count')
analysis_df['repeat_customer'] = (customer_order_counts > 1).astype(int)

print(f"\n✓ Added repeat_customer column to analysis_df")
print(f"  - Repeat customers: {analysis_df['repeat_customer'].sum():,} ({analysis_df['repeat_customer'].mean()*100:.1f}%)")
print(f"  - One-time customers: {(analysis_df['repeat_customer']==0).sum():,} ({(1-analysis_df['repeat_customer'].mean())*100:.1f}%)")

print(f"\n✓ Final analysis_df shape: {analysis_df.shape}")
print(f"  - Ready for business impact analysis in Cell 60")

# Delays Performance

In [0]:
"""
T-test: relationship between delivery delay and customer review score (Olist).

Two independent-samples t-tests (Welch, unequal variance):
  Test A - review_score for LATE vs ON-TIME orders
  Test B - delay_days for LOW-rated (1-2) vs HIGH-rated (4-5) orders
"""

import numpy as np
import pandas as pd
from scipy import stats

# ============================================================
# 1. LOAD AND BUILD THE ANALYSIS TABLE
# ============================================================

orders = olist_orders_dataset_df.copy()
reviews = olist_order_reviews_dataset_df.copy()

for column in ["order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

# Keep only delivered orders that have BOTH dates needed for delay_days.
# (Previously only order_delivered_customer_date was required; a missing
# order_estimated_delivery_date silently produced delay_days = NaN and,
# because NaT > 0 is False, got miscounted as "on-time" downstream.)
delivered_orders = orders[
    (orders["order_status"] == "delivered")
    & orders["order_delivered_customer_date"].notna()
    & orders["order_estimated_delivery_date"].notna()
].copy()

# Positive = late, negative = early
delivered_orders["delay_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

delivered_orders["late_order"] = (delivered_orders["delay_days"] > 0).astype(int)

# One review per order (keep the most recent if duplicated)
reviews["review_creation_date"] = pd.to_datetime(
    reviews["review_creation_date"], errors="coerce"
)
reviews = (
    reviews.sort_values("review_creation_date")
    .drop_duplicates(subset="order_id", keep="last")
)

df = delivered_orders.merge(
    reviews[["order_id", "review_score"]], on="order_id", how="inner"
)

# Sanity check: delay_days should now be fully populated for every row in df.
# If this ever prints > 0, something upstream let a NaT through and every
# n/mean/SD below would be silently wrong for the affected group.
n_missing_delay = df["delay_days"].isna().sum()
assert n_missing_delay == 0, (
    f"{n_missing_delay} rows have NaN delay_days after the notna() filter "
    "— check for unexpected date parsing failures."
)

print("=" * 62)
print("SAMPLE")
print("=" * 62)
print(f"Delivered orders discarded (missing dates): "
      f"{len(orders[orders['order_status'] == 'delivered']) - len(delivered_orders):,}")
print(f"Delivered orders with a review : {len(df):,}")
print(f"Missing delay_days in final df : {n_missing_delay:,}")
print(f"Late orders                    : {df['late_order'].sum():,} "
      f"({df['late_order'].mean() * 100:.1f}%)")
print(f"On-time orders                 : {(1 - df['late_order']).sum():,} "
      f"({(1 - df['late_order']).mean() * 100:.1f}%)")


def welch_report(name, group1_name, group1, group2_name, group2):
    t_stat, p_value = stats.ttest_ind(group1, group2, equal_var=False)

    n1, n2 = len(group1), len(group2)
    m1, m2 = group1.mean(), group2.mean()
    s1, s2 = group1.std(ddof=1), group2.std(ddof=1)

    # Welch degrees of freedom (Welch-Satterthwaite) - matches what
    # ttest_ind(equal_var=False) uses internally for p_value.
    v1, v2 = s1 ** 2 / n1, s2 ** 2 / n2
    dof = (v1 + v2) ** 2 / (v1 ** 2 / (n1 - 1) + v2 ** 2 / (n2 - 1))

    # Pooled-SD Cohen's d - standard, but note it assumes equal variance,
    # which conflicts with the Welch test's own assumption. Reported
    # alongside Glass's delta (below) for consistency with unequal variances.
    pooled_sd = np.sqrt(((n1 - 1) * s1 ** 2 + (n2 - 1) * s2 ** 2) / (n1 + n2 - 2))
    cohens_d = (m1 - m2) / pooled_sd

    # Glass's delta: uses only group2's SD as the reference. Appropriate
    # when variances differ a lot (as they do here) and one group is the
    # natural baseline (on-time / high-rated).
    glass_delta = (m1 - m2) / s2

    # 95% CI for the difference in means, using the same Welch dof as the test.
    se_diff = np.sqrt(v1 + v2)
    t_crit = stats.t.ppf(0.975, dof)
    ci_low = (m1 - m2) - t_crit * se_diff
    ci_high = (m1 - m2) + t_crit * se_diff

    print()
    print("=" * 62)
    print(name)
    print("=" * 62)
    print(f"{group1_name:<22} n = {n1:>7,}   mean = {m1:>8.3f}   sd = {s1:>7.3f}")
    print(f"{group2_name:<22} n = {n2:>7,}   mean = {m2:>8.3f}   sd = {s2:>7.3f}")
    print("-" * 62)
    print(f"Difference in means : {m1 - m2:.3f}")
    print(f"95% CI of difference: [{ci_low:.3f}, {ci_high:.3f}]")
    print(f"t-statistic         : {t_stat:.3f}")
    print(f"degrees of freedom  : {dof:.1f}")
    print(f"p-value             : {p_value:.3e}"
          + ("  (p < 0.001)" if p_value < 0.001 else ""))
    print(f"Cohen's d (pooled)  : {cohens_d:.3f}")
    print(f"Glass's delta       : {glass_delta:.3f}  (uses {group2_name} SD only)")
    print(f"Decision (a = 0.05) : "
          f"{'REJECT the null hypothesis' if p_value < 0.05 else 'fail to reject the null'}")


# ============================================================
# 2. TEST A - review score: late vs on-time
# ============================================================
welch_report(
    "TEST A: review_score  ~  late vs on-time delivery",
    "Late orders",
    df.loc[df["late_order"] == 1, "review_score"],
    "On-time orders",
    df.loc[df["late_order"] == 0, "review_score"],
)

# ============================================================
# 3. TEST B - delay days: low-rated vs high-rated
# ============================================================
welch_report(
    "TEST B: delay_days  ~  low (1-2 stars) vs high (4-5 stars)",
    "Low-rated (1-2)",
    df.loc[df["review_score"] <= 2, "delay_days"],
    "High-rated (4-5)",
    df.loc[df["review_score"] >= 4, "delay_days"],
)

# ============================================================
# 4. SUPPORTING DESCRIPTIVES
# ============================================================
print()
print("=" * 62)
print("MEAN REVIEW SCORE AND LATE RATE BY DELAY BUCKET")
print("=" * 62)

buckets = pd.cut(
    df["delay_days"],
    bins=[-np.inf, -10, -5, 0, 5, 10, np.inf],
    labels=["> 10d early", "5-10d early", "0-5d early",
            "0-5d late", "5-10d late", "> 10d late"],
)

summary = df.groupby(buckets, observed=True).agg(
    orders=("review_score", "size"),
    mean_review_score=("review_score", "mean"),
    pct_1_star=("review_score", lambda s: (s == 1).mean() * 100),
)
print(summary.round(2).to_string())

print()
print("Mean review score by delay bucket, correlation check:")
r, p_r = stats.pearsonr(df["delay_days"], df["review_score"])
rho, p_rho = stats.spearmanr(df["delay_days"], df["review_score"])
print(f"  Pearson  r = {r:.3f}  (p = {p_r:.3e})")
print(f"  Spearman rho = {rho:.3f}  (p = {p_rho:.3e})")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6796148409518962>, line 17
     11 from scipy import stats
     13 # ============================================================
     14 # 1. LOAD AND BUILD THE ANALYSIS TABLE
     15 # ============================================================
---> 17 orders = olist_orders_dataset_df.copy()
     18 reviews = olist_order_reviews_dataset_df.copy()
     20 for column in ["order_delivered_customer_date", "order_estimated_delivery_date"]:

NameError: name 'olist_orders_dataset_df' is not defined

In [0]:
from scipy import stats
import pandas as pd

# ------------------------------------------------------------
# Chi-square test of independence
# delivery_status (Delayed vs On time or early) x repeat_purchase (0/1)
# ------------------------------------------------------------

contingency_table = pd.crosstab(
    repeat_analysis_df["delivery_status"],
    repeat_analysis_df["repeat_purchase"]
)

chi2_stat, chi2_p_value, chi2_dof, expected_counts = stats.chi2_contingency(
    contingency_table
)

print("=" * 60)
print("CHI-SQUARE TEST OF INDEPENDENCE")
print("=" * 60)
print("Contingency table (rows = delivery status, cols = repeat_purchase):")
print(contingency_table)
print("-" * 60)
print(f"Chi-square statistic: {chi2_stat:.3f}")
print(f"Degrees of freedom: {chi2_dof}")
print(f"p-value: {chi2_p_value:.3e}")
print(
    "Decision (alpha = 0.05): " +
    ("reject the null hypothesis" if chi2_p_value < 0.05 else "fail to reject the null")
)
print()


# ------------------------------------------------------------
# Two-proportion z-test
# ------------------------------------------------------------

delayed_group = repeat_analysis_df.loc[
    repeat_analysis_df["delivery_status"] == "Delayed",
    "repeat_purchase"
]

on_time_group = repeat_analysis_df.loc[
    repeat_analysis_df["delivery_status"] == "On time or early",
    "repeat_purchase"
]

n1, n2 = len(delayed_group), len(on_time_group)
p1, p2 = delayed_group.mean(), on_time_group.mean()

# Pooled proportion under the null (no difference between groups)
pooled_p = (delayed_group.sum() + on_time_group.sum()) / (n1 + n2)
pooled_se = np.sqrt(
    pooled_p * (1 - pooled_p) * (1 / n1 + 1 / n2)
)

z_stat = (p1 - p2) / pooled_se
z_p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

# 95% CI for the difference in proportions (unpooled standard error)
unpooled_se = np.sqrt(
    p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2
)
ci_low = (p1 - p2) - 1.96 * unpooled_se
ci_high = (p1 - p2) + 1.96 * unpooled_se

# Cohen's h effect size for two proportions
cohens_h = 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

print("=" * 60)
print("TWO-PROPORTION Z-TEST")
print("=" * 60)
print(f"Delayed: n = {n1:,}, repeat purchase rate = {p1:.4f}")
print(f"On time or early: n = {n2:,}, repeat purchase rate = {p2:.4f}")
print("-" * 60)
print(f"Difference in rate: {p1 - p2:.4f}")
print(f"95% CI of difference: [{ci_low:.4f}, {ci_high:.4f}]")
print(f"z-statistic: {z_stat:.3f}")
print(f"p-value: {z_p_value:.3e}")
print(f"Cohen's h: {cohens_h:.3f}")
print(
    "Decision (alpha = 0.05): " +
    ("reject the null hypothesis" if z_p_value < 0.05 else "fail to reject the null")
)
print()


# ------------------------------------------------------------
# Point-biserial correlation: delay_days (continuous) vs repeat_purchase (0/1)
# ------------------------------------------------------------

point_biserial_r, point_biserial_p = stats.pointbiserialr(
    repeat_analysis_df["repeat_purchase"],
    repeat_analysis_df["delay_days"]
)

print("=" * 60)
print("POINT-BISERIAL CORRELATION")
print("=" * 60)
print(f"r = {point_biserial_r:.3f}")
print(f"p-value = {point_biserial_p:.3e}")


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5212843221354972>, line 10
      2 import pandas as pd
      4 # ------------------------------------------------------------
      5 # Chi-square test of independence
      6 # delivery_status (Delayed vs On time or early) x repeat_purchase (0/1)
      7 # ------------------------------------------------------------
      9 contingency_table = pd.crosstab(
---> 10     repeat_analysis_df["delivery_status"],
     11     repeat_analysis_df["repeat_purchase"]
     12 )
     14 chi2_stat, chi2_p_value, chi2_dof, expected_counts = stats.chi2_contingency(
     15     contingency_table
     16 )
     18 print("=" * 60)

NameError: name 'repeat_analysis_df' is not defined

In [0]:
# Keep only late orders
late_orders_df = analysis_df[
    analysis_df["delay_days"] > 0
].copy()

# Percentage of all orders that were late
late_order_percentage = (
    len(late_orders_df) / len(analysis_df)
) * 100

# Average delay using only late orders
average_lateness_days = late_orders_df["delay_days"].mean()

print("LATE ORDER ANALYSIS")
print("------------------------------------")
print(f"Late orders: {len(late_orders_df):,}")
print(f"Percentage of orders that were late: {late_order_percentage:.2f}%")
print(f"Average lateness among late orders: {average_lateness_days:.2f} days")

In [0]:
# Average review score per order
reviews_per_order = (
    olist_order_reviews_dataset_df
    .groupby("order_id", as_index=False)["review_score"]
    .mean()
)

# Add review scores to the delivery analysis
review_analysis_df = analysis_df[
    ["order_id", "delay_days"]
].merge(
    reviews_per_order,
    on="order_id",
    how="inner"
)

# Create delivery status
review_analysis_df["delivery_status"] = np.where(
    review_analysis_df["delay_days"] > 0,
    "Delayed",
    "On time or early"
)

# Calculate average review score for each group
average_review_scores = (
    review_analysis_df
    .groupby("delivery_status")["review_score"]
    .agg(["mean", "count"])
    .round(2)
)

average_review_scores.columns = [
    "average_review_score",
    "number_of_reviews"
]

print("AVERAGE REVIEW SCORE BY DELIVERY STATUS")
print("------------------------------------------")
print(average_review_scores)

In [0]:
# ============================================================
# REPEAT PURCHASE RATE BY DELIVERY STATUS
# ============================================================

# Connect each order to the permanent customer identifier
customer_order_history = (
    olist_orders_dataset_df[
        [
            "order_id",
            "customer_id",
            "order_purchase_timestamp",
            "order_status"
        ]
    ]
    .merge(
        olist_customers_dataset_df[
            ["customer_id", "customer_unique_id"]
        ],
        on="customer_id",
        how="left"
    )
)

# Convert purchase timestamp to datetime
customer_order_history["order_purchase_timestamp"] = pd.to_datetime(
    customer_order_history["order_purchase_timestamp"]
)

# Exclude cancelled and unavailable orders
customer_order_history = customer_order_history[
    ~customer_order_history["order_status"].isin(
        ["canceled", "unavailable"]
    )
].copy()

# Sort each customer's orders chronologically
customer_order_history = customer_order_history.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"]
)

# Find the customer's next purchase
customer_order_history["next_purchase_date"] = (
    customer_order_history
    .groupby("customer_unique_id")["order_purchase_timestamp"]
    .shift(-1)
)

# 1 means the customer made another purchase later
customer_order_history["repeat_purchase"] = (
    customer_order_history["next_purchase_date"].notna()
).astype(int)

# Add delivery delay information
repeat_analysis_df = analysis_df[
    ["order_id", "delay_days"]
].merge(
    customer_order_history[
        ["order_id", "customer_unique_id", "repeat_purchase"]
    ],
    on="order_id",
    how="inner"
)

# Separate delayed orders from on-time or early orders
repeat_analysis_df["delivery_status"] = np.where(
    repeat_analysis_df["delay_days"] > 0,
    "Delayed",
    "On time or early"
)

# Calculate repeat purchase rate
repeat_purchase_results = (
    repeat_analysis_df
    .groupby("delivery_status")
    .agg(
        number_of_orders=("order_id", "count"),
        repeat_purchases=("repeat_purchase", "sum"),
        repeat_purchase_rate=("repeat_purchase", "mean")
    )
    .reset_index()
)

repeat_purchase_results["repeat_purchase_rate"] = (
    repeat_purchase_results["repeat_purchase_rate"] * 100
).round(2)

print("REPEAT PURCHASE RATE BY DELIVERY STATUS")
print("----------------------------------------")
print(repeat_purchase_results)

In [0]:
# If Dash is not installed, run this once:
# %pip install dash

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from dash import (
    Dash,
    dcc,
    html,
    Input,
    Output,
    dash_table
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix
)

# ============================================================
# 1. MODEL RESULTS
# ============================================================

def create_model_result(
    model,
    feature_set,
    y_true,
    predictions
):
    return {
        "model": model,
        "feature_set": feature_set,
        "accuracy": accuracy_score(
            y_true,
            predictions
        ),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        )
    }


model_results_df = pd.DataFrame([
    create_model_result(
        "Logistic Regression",
        "Original",
        Y_test,
        delay_predictions
    ),

    create_model_result(
        "Logistic Regression",
        "Transformed",
        Y_test_log,
        log_predictions
    ),

    create_model_result(
        "Random Forest",
        "Original",
        Y_test,
        rf_predictions
    ),

    create_model_result(
        "Random Forest",
        "Transformed",
        Y_test_rf_log,
        transformed_rf_predictions
    ),

    create_model_result(
        "Neural Network",
        "Original",
        Y_test,
        nn_predictions
    )
])

model_results_df["model_label"] = (
    model_results_df["model"] +
    " - " +
    model_results_df["feature_set"]
)


# Confusion matrices for each model

model_confusion_matrices = {
    ("Logistic Regression", "Original"):
        confusion_matrix(Y_test, delay_predictions),

    ("Logistic Regression", "Transformed"):
        confusion_matrix(Y_test_log, log_predictions),

    ("Random Forest", "Original"):
        confusion_matrix(Y_test, rf_predictions),

    ("Random Forest", "Transformed"):
        confusion_matrix(
            Y_test_rf_log,
            transformed_rf_predictions
        ),

    ("Neural Network", "Original"):
        confusion_matrix(Y_test, nn_predictions)
}


# ============================================================
# 2. PREPARE BUSINESS KPI DATA
# ============================================================

business_df = analysis_df.copy()


# Add permanent customer identifier and city

customer_information = (
    olist_customers_dataset_df[
        [
            "customer_id",
            "customer_unique_id",
            "customer_city"
        ]
    ]
    .drop_duplicates("customer_id")
)

business_df = business_df.merge(
    customer_information,
    on="customer_id",
    how="left"
)


# Standardise city names for the dashboard

business_df["customer_city"] = (
    business_df["customer_city"]
    .str.title()
)


# Add the average review score for each order

reviews_per_order = (
    olist_order_reviews_dataset_df
    .groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean")
    )
)

business_df = business_df.merge(
    reviews_per_order,
    on="order_id",
    how="left"
)


# ============================================================
# 3. ENGINEER REPEAT-PURCHASE FEATURE
# ============================================================

order_history = olist_orders_dataset_df[
    [
        "order_id",
        "customer_id",
        "order_purchase_timestamp",
        "order_status"
    ]
].copy()

order_history["order_purchase_timestamp"] = pd.to_datetime(
    order_history["order_purchase_timestamp"],
    errors="coerce"
)

order_history = order_history[
    ~order_history["order_status"].isin(
        ["canceled", "unavailable"]
    )
].copy()

order_history = order_history.merge(
    olist_customers_dataset_df[
        ["customer_id", "customer_unique_id"]
    ],
    on="customer_id",
    how="left"
)

order_history = order_history.sort_values(
    [
        "customer_unique_id",
        "order_purchase_timestamp"
    ]
)

order_history["next_purchase_date"] = (
    order_history
    .groupby("customer_unique_id")[
        "order_purchase_timestamp"
    ]
    .shift(-1)
)

order_history["repeat_purchase"] = (
    order_history["next_purchase_date"].notna()
).astype(int)

business_df = business_df.merge(
    order_history[
        ["order_id", "repeat_purchase"]
    ],
    on="order_id",
    how="left"
)

business_df["repeat_purchase"] = (
    business_df["repeat_purchase"]
    .fillna(0)
    .astype(int)
)


# Use consistent dashboard labels

business_df["delivery_status"] = np.where(
    business_df["delay_days"] > 0,
    "Delayed",
    "On time or early"
)

business_df["order_purchase_timestamp"] = pd.to_datetime(
    business_df["order_purchase_timestamp"],
    errors="coerce"
)


# ============================================================
# 4. DATASETS FOR DATA-QUALITY TAB
# ============================================================

quality_datasets = {
    "Customers":
        olist_customers_dataset_df,

    "Geolocation":
        olist_geolocation_dataset_df,

    "Orders":
        olist_orders_dataset_df,

    "Order Items":
        olist_order_items_dataset_df,

    "Order Payments":
        olist_order_payments_dataset_df,

    "Order Reviews":
        olist_order_reviews_dataset_df,

    "Products":
        olist_products_dataset_df,

    "Sellers":
        olist_sellers_dataset_df,

    "Product Category Translation":
        olist_product_category_translation_df
}


# ============================================================
# 5. MODEL COMPARISON GRAPH
# ============================================================

model_results_long = model_results_df.melt(
    id_vars=[
        "model",
        "feature_set",
        "model_label"
    ],
    value_vars=[
        "accuracy",
        "precision",
        "recall"
    ],
    var_name="metric",
    value_name="score"
)

model_comparison_figure = px.bar(
    model_results_long,
    x="model_label",
    y="score",
    color="metric",
    barmode="group",
    labels={
        "model_label": "Model",
        "score": "Score",
        "metric": "Metric"
    },
    title="Comparison of All Delay-Prediction Models",
    template="simple_white",
    color_discrete_map={
        "accuracy": "#2563EB",
        "precision": "#F59E0B",
        "recall": "#10B981"
    }
)

model_comparison_figure.update_yaxes(
    range=[0, 1],
    tickformat=".0%"
)

model_comparison_figure.update_layout(
    xaxis_tickangle=-20,
    legend_title_text=""
)

In [0]:
# ============================================================
# DASHBOARD STYLES
# ============================================================

PAGE_STYLE = {
    "fontFamily": "Arial, sans-serif",
    "backgroundColor": "#F3F6FA",
    "minHeight": "100vh",
    "padding": "25px"
}

SECTION_STYLE = {
    "backgroundColor": "white",
    "padding": "22px",
    "marginTop": "18px",
    "borderRadius": "12px",
    "boxShadow": "0 2px 8px rgba(0,0,0,0.08)"
}

FILTER_GRID_STYLE = {
    "display": "grid",
    "gridTemplateColumns": "repeat(auto-fit, minmax(190px, 1fr))",
    "gap": "15px",
    "marginBottom": "20px"
}

KPI_GRID_STYLE = {
    "display": "grid",
    "gridTemplateColumns": "repeat(auto-fit, minmax(160px, 1fr))",
    "gap": "15px",
    "marginBottom": "20px"
}

KPI_CARD_STYLE = {
    "backgroundColor": "#F8FAFC",
    "padding": "18px",
    "borderRadius": "10px",
    "borderLeft": "5px solid #2563EB",
    "textAlign": "center"
}

KPI_VALUE_STYLE = {
    "fontSize": "27px",
    "fontWeight": "bold",
    "color": "#1E3A5F",
    "margin": "5px"
}

GRAPH_GRID_STYLE = {
    "display": "grid",
    "gridTemplateColumns": "repeat(auto-fit, minmax(400px, 1fr))",
    "gap": "18px"
}


def create_kpi_card(title, component_id):
    return html.Div(
        [
            html.P(
                title,
                style={
                    "color": "#64748B",
                    "margin": "0"
                }
            ),
            html.P(
                id=component_id,
                style=KPI_VALUE_STYLE
            )
        ],
        style=KPI_CARD_STYLE
    )


# ============================================================
# FILTER OPTIONS
# ============================================================

state_options = [
    {"label": "All states", "value": "All"}
] + [
    {"label": state, "value": state}
    for state in sorted(
        business_df["customer_state"]
        .dropna()
        .unique()
    )
]

minimum_date = (
    business_df["order_purchase_timestamp"]
    .min()
    .date()
)

maximum_date = (
    business_df["order_purchase_timestamp"]
    .max()
    .date()
)


# ============================================================
# CREATE DASH APPLICATION
# ============================================================

app = Dash(__name__)

app.layout = html.Div(
    [
        # Dashboard heading

        html.Div(
            [
                html.H1(
                    "Olist Delivery Performance Dashboard",
                    style={
                        "marginBottom": "5px",
                        "color": "#1E3A5F"
                    }
                ),
                html.P(
                    "Model performance, regional business KPIs "
                    "and dataset quality",
                    style={
                        "color": "#64748B",
                        "fontSize": "17px"
                    }
                )
            ],
            style={
                "backgroundColor": "white",
                "padding": "22px",
                "borderRadius": "12px",
                "borderTop": "6px solid #2563EB"
            }
        ),

        dcc.Tabs(
            [
                # =================================================
                # TAB 1 — MODEL SELECTION
                # =================================================

                dcc.Tab(
                    label="Model Selection",
                    children=[
                        html.Div(
                            [
                                html.H2(
                                    "Delay Prediction Models"
                                ),

                                html.P(
                                    "Select a model and choose whether "
                                    "the original or transformed features "
                                    "were used."
                                ),

                                html.Div(
                                    [
                                        html.Div(
                                            [
                                                html.Label(
                                                    "Model"
                                                ),
                                                dcc.Dropdown(
                                                    id="model-selector",
                                                    options=[
                                                        {
                                                            "label":
                                                            "Logistic Regression",
                                                            "value":
                                                            "Logistic Regression"
                                                        },
                                                        {
                                                            "label":
                                                            "Random Forest",
                                                            "value":
                                                            "Random Forest"
                                                        },
                                                        {
                                                            "label":
                                                            "Neural Network",
                                                            "value":
                                                            "Neural Network"
                                                        }
                                                    ],
                                                    value="Random Forest",
                                                    clearable=False
                                                )
                                            ]
                                        ),

                                        html.Div(
                                            [
                                                html.Label(
                                                    "Feature set"
                                                ),
                                                dcc.Dropdown(
                                                    id="transformation-selector",
                                                    value="Original",
                                                    clearable=False
                                                )
                                            ]
                                        )
                                    ],
                                    style=FILTER_GRID_STYLE
                                ),

                                html.H3(
                                    id="selected-model-title",
                                    style={"color": "#1E3A5F"}
                                ),

                                html.Div(
                                    [
                                        create_kpi_card(
                                            "Accuracy",
                                            "model-accuracy"
                                        ),
                                        create_kpi_card(
                                            "Precision",
                                            "model-precision"
                                        ),
                                        create_kpi_card(
                                            "Recall",
                                            "model-recall"
                                        )
                                    ],
                                    style=KPI_GRID_STYLE
                                ),

                                html.Div(
                                    [
                                        dcc.Graph(
                                            id="confusion-matrix-graph"
                                        ),
                                        dcc.Graph(
                                            figure=model_comparison_figure
                                        )
                                    ],
                                    style=GRAPH_GRID_STYLE
                                )
                            ],
                            style=SECTION_STYLE
                        )
                    ]
                ),

                # =================================================
                # TAB 2 — BUSINESS KPIs
                # =================================================

                dcc.Tab(
                    label="Business KPIs",
                    children=[
                        html.Div(
                            [
                                html.H2(
                                    "Regional Delivery Performance"
                                ),

                                html.P(
                                    "Use the filters to compare customers, "
                                    "delays, reviews and repeat purchasing."
                                ),

                                html.Div(
                                    [
                                        html.Div(
                                            [
                                                html.Label("State"),
                                                dcc.Dropdown(
                                                    id="state-filter",
                                                    options=state_options,
                                                    value="All",
                                                    clearable=False
                                                )
                                            ]
                                        ),

                                        html.Div(
                                            [
                                                html.Label("City"),
                                                dcc.Dropdown(
                                                    id="city-filter",
                                                    value="All",
                                                    clearable=False
                                                )
                                            ]
                                        ),

                                        html.Div(
                                            [
                                                html.Label(
                                                    "Delivery status"
                                                ),
                                                dcc.Dropdown(
                                                    id="delivery-filter",
                                                    options=[
                                                        {
                                                            "label":
                                                            "All orders",
                                                            "value":
                                                            "All"
                                                        },
                                                        {
                                                            "label":
                                                            "On time or early",
                                                            "value":
                                                            "On time or early"
                                                        },
                                                        {
                                                            "label":
                                                            "Delayed",
                                                            "value":
                                                            "Delayed"
                                                        }
                                                    ],
                                                    value="All",
                                                    clearable=False
                                                )
                                            ]
                                        ),

                                        html.Div(
                                            [
                                                html.Label(
                                                    "Purchase dates"
                                                ),
                                                dcc.DatePickerRange(
                                                    id="date-filter",
                                                    min_date_allowed=minimum_date,
                                                    max_date_allowed=maximum_date,
                                                    start_date=minimum_date,
                                                    end_date=maximum_date
                                                )
                                            ]
                                        )
                                    ],
                                    style=FILTER_GRID_STYLE
                                ),

                                html.Div(
                                    [
                                        create_kpi_card(
                                            "Delivered orders",
                                            "kpi-orders"
                                        ),
                                        create_kpi_card(
                                            "Unique customers",
                                            "kpi-customers"
                                        ),
                                        create_kpi_card(
                                            "Late-order rate",
                                            "kpi-late-rate"
                                        ),
                                        create_kpi_card(
                                            "Average lateness",
                                            "kpi-average-late"
                                        ),
                                        create_kpi_card(
                                            "Average review score",
                                            "kpi-review"
                                        ),
                                        create_kpi_card(
                                            "Repeat-purchase rate",
                                            "kpi-repeat"
                                        )
                                    ],
                                    style=KPI_GRID_STYLE
                                ),

                                html.Div(
                                    [
                                        dcc.Graph(
                                            id="city-delay-graph"
                                        ),
                                        dcc.Graph(
                                            id="review-status-graph"
                                        )
                                    ],
                                    style=GRAPH_GRID_STYLE
                                )
                            ],
                            style=SECTION_STYLE
                        )
                    ]
                ),

                # =================================================
                # TAB 3 — DATA QUALITY
                # =================================================

                dcc.Tab(
                    label="Data Quality",
                    children=[
                        html.Div(
                            [
                                html.H2(
                                    "Dataset Quality Explorer"
                                ),

                                html.P(
                                    "Select one of the nine Olist tables "
                                    "to examine its structure and quality."
                                ),

                                html.Div(
                                    [
                                        html.Div(
                                            [
                                                html.Label("Dataset"),
                                                dcc.Dropdown(
                                                    id="dataset-selector",
                                                    options=[
                                                        {
                                                            "label": name,
                                                            "value": name
                                                        }
                                                        for name in
                                                        quality_datasets.keys()
                                                    ],
                                                    value="Orders",
                                                    clearable=False
                                                )
                                            ]
                                        )
                                    ],
                                    style=FILTER_GRID_STYLE
                                ),

                                html.Div(
                                    [
                                        create_kpi_card(
                                            "Rows",
                                            "quality-rows"
                                        ),
                                        create_kpi_card(
                                            "Columns",
                                            "quality-columns"
                                        ),
                                        create_kpi_card(
                                            "Missing values",
                                            "quality-missing"
                                        ),
                                        create_kpi_card(
                                            "Missing cells",
                                            "quality-missing-percent"
                                        ),
                                        create_kpi_card(
                                            "Duplicate rows",
                                            "quality-duplicates"
                                        )
                                    ],
                                    style=KPI_GRID_STYLE
                                ),

                                dcc.Graph(
                                    id="missing-values-graph"
                                ),

                                html.H3(
                                    "Column-level quality"
                                ),

                                dash_table.DataTable(
                                    id="quality-table",
                                    page_size=12,
                                    sort_action="native",
                                    filter_action="native",
                                    style_table={
                                        "overflowX": "auto"
                                    },
                                    style_header={
                                        "backgroundColor": "#1E3A5F",
                                        "color": "white",
                                        "fontWeight": "bold"
                                    },
                                    style_cell={
                                        "padding": "9px",
                                        "textAlign": "left",
                                        "fontFamily": "Arial",
                                        "fontSize": "13px"
                                    },
                                    style_data_conditional=[
                                        {
                                            "if": {
                                                "filter_query":
                                                "{missing_values} > 0"
                                            },
                                            "backgroundColor": "#FEF3C7"
                                        }
                                    ]
                                )
                            ],
                            style=SECTION_STYLE
                        )
                    ]
                )
            ],
            style={"marginTop": "18px"}
        )
    ],
    style=PAGE_STYLE
)

In [0]:
# ============================================================
# 1. MODEL-SELECTION CALLBACKS
# ============================================================

@app.callback(
    Output(
        "transformation-selector",
        "options"
    ),
    Output(
        "transformation-selector",
        "value"
    ),
    Input(
        "model-selector",
        "value"
    )
)
def update_transformation_options(selected_model):

    # A transformed neural network was not trained
    if selected_model == "Neural Network":
        options = [
            {
                "label": "Original features",
                "value": "Original"
            }
        ]

    else:
        options = [
            {
                "label": "Original features",
                "value": "Original"
            },
            {
                "label": "Transformed features",
                "value": "Transformed"
            }
        ]

    return options, "Original"


@app.callback(
    Output(
        "selected-model-title",
        "children"
    ),
    Output(
        "model-accuracy",
        "children"
    ),
    Output(
        "model-precision",
        "children"
    ),
    Output(
        "model-recall",
        "children"
    ),
    Output(
        "confusion-matrix-graph",
        "figure"
    ),
    Input(
        "model-selector",
        "value"
    ),
    Input(
        "transformation-selector",
        "value"
    )
)
def update_model_results(
    selected_model,
    selected_feature_set
):

    selected_result = model_results_df[
        (model_results_df["model"] == selected_model) &
        (
            model_results_df["feature_set"] ==
            selected_feature_set
        )
    ]

    # Safety check for a combination that does not exist
    if selected_result.empty:
        selected_feature_set = "Original"

        selected_result = model_results_df[
            (model_results_df["model"] == selected_model) &
            (
                model_results_df["feature_set"] ==
                selected_feature_set
            )
        ]

    selected_result = selected_result.iloc[0]

    confusion = model_confusion_matrices[
        (
            selected_model,
            selected_feature_set
        )
    ]

    confusion_figure = px.imshow(
        confusion,
        text_auto=True,
        color_continuous_scale="Blues",
        x=[
            "Predicted on time",
            "Predicted delayed"
        ],
        y=[
            "Actually on time",
            "Actually delayed"
        ],
        labels={
            "x": "Prediction",
            "y": "Actual result",
            "color": "Orders"
        },
        title="Confusion Matrix"
    )

    confusion_figure.update_layout(
        template="simple_white"
    )

    title = (
        f"{selected_model} — "
        f"{selected_feature_set} features"
    )

    return (
        title,
        f"{selected_result['accuracy']:.1%}",
        f"{selected_result['precision']:.1%}",
        f"{selected_result['recall']:.1%}",
        confusion_figure
    )


# ============================================================
# 2. CITY OPTIONS CALLBACK
# ============================================================

@app.callback(
    Output(
        "city-filter",
        "options"
    ),
    Output(
        "city-filter",
        "value"
    ),
    Input(
        "state-filter",
        "value"
    )
)
def update_city_options(selected_state):

    filtered_cities = business_df.copy()

    if selected_state != "All":
        filtered_cities = filtered_cities[
            filtered_cities["customer_state"] ==
            selected_state
        ]

    cities = sorted(
        filtered_cities["customer_city"]
        .dropna()
        .unique()
    )

    options = [
        {
            "label": "All cities",
            "value": "All"
        }
    ] + [
        {
            "label": city,
            "value": city
        }
        for city in cities
    ]

    return options, "All"


# ============================================================
# 3. BUSINESS KPI CALLBACK
# ============================================================

@app.callback(
    Output("kpi-orders", "children"),
    Output("kpi-customers", "children"),
    Output("kpi-late-rate", "children"),
    Output("kpi-average-late", "children"),
    Output("kpi-review", "children"),
    Output("kpi-repeat", "children"),
    Output("city-delay-graph", "figure"),
    Output("review-status-graph", "figure"),

    Input("state-filter", "value"),
    Input("city-filter", "value"),
    Input("delivery-filter", "value"),
    Input("date-filter", "start_date"),
    Input("date-filter", "end_date")
)
def update_business_kpis(
    selected_state,
    selected_city,
    selected_delivery_status,
    start_date,
    end_date
):

    filtered_df = business_df.copy()

    if selected_state != "All":
        filtered_df = filtered_df[
            filtered_df["customer_state"] ==
            selected_state
        ]

    if selected_city != "All":
        filtered_df = filtered_df[
            filtered_df["customer_city"] ==
            selected_city
        ]

    if selected_delivery_status != "All":
        filtered_df = filtered_df[
            filtered_df["delivery_status"] ==
            selected_delivery_status
        ]

    if start_date is not None:
        filtered_df = filtered_df[
            filtered_df["order_purchase_timestamp"] >=
            pd.to_datetime(start_date)
        ]

    if end_date is not None:
        filtered_df = filtered_df[
            filtered_df["order_purchase_timestamp"] <
            pd.to_datetime(end_date) +
            pd.Timedelta(days=1)
        ]

    # KPI calculations

    number_of_orders = len(filtered_df)

    unique_customers = (
        filtered_df["customer_unique_id"]
        .nunique()
    )

    if number_of_orders > 0:
        late_rate = (
            filtered_df["late_order"].mean() * 100
        )

        repeat_rate = (
            filtered_df["repeat_purchase"].mean() * 100
        )

    else:
        late_rate = np.nan
        repeat_rate = np.nan

    # Average lateness excludes early/on-time orders

    late_orders = filtered_df[
        filtered_df["delay_days"] > 0
    ]

    average_lateness = (
        late_orders["delay_days"].mean()
    )

    average_review = (
        filtered_df["review_score"].mean()
    )

    # City-level summary

    city_summary = (
        filtered_df
        .groupby("customer_city")
        .agg(
            order_count=("order_id", "count"),
            late_order_rate=("late_order", "mean"),
            average_review=("review_score", "mean")
        )
        .reset_index()
    )

    city_summary["late_order_rate"] = (
        city_summary["late_order_rate"] * 100
    )

    # Show the 15 cities with the most orders

    city_summary = (
        city_summary
        .sort_values(
            "order_count",
            ascending=False
        )
        .head(15)
        .sort_values(
            "late_order_rate",
            ascending=False
        )
    )

    city_delay_figure = px.bar(
        city_summary,
        x="customer_city",
        y="late_order_rate",
        color="average_review",
        hover_data=["order_count"],
        labels={
            "customer_city": "Customer city",
            "late_order_rate": "Late orders (%)",
            "average_review": "Average review",
            "order_count": "Orders"
        },
        title="Late-Order Rate in the Largest Selected Cities",
        template="simple_white",
        color_continuous_scale="RdYlGn"
    )

    city_delay_figure.update_layout(
        xaxis_tickangle=-30
    )

    # Review score comparison

    review_summary = (
        filtered_df
        .groupby("delivery_status")
        .agg(
            average_review_score=(
                "review_score",
                "mean"
            ),
            number_of_reviews=(
                "review_score",
                "count"
            )
        )
        .reset_index()
    )

    review_figure = px.bar(
        review_summary,
        x="delivery_status",
        y="average_review_score",
        color="delivery_status",
        hover_data=["number_of_reviews"],
        labels={
            "delivery_status": "Delivery status",
            "average_review_score":
            "Average review score",
            "number_of_reviews":
            "Number of reviews"
        },
        title="Average Review Score by Delivery Status",
        template="simple_white",
        color_discrete_map={
            "On time or early": "#10B981",
            "Delayed": "#EF4444"
        }
    )

    review_figure.update_yaxes(
        range=[0, 5]
    )

    # Format KPI values

    orders_text = f"{number_of_orders:,}"
    customers_text = f"{unique_customers:,}"

    late_text = (
        f"{late_rate:.2f}%"
        if pd.notna(late_rate)
        else "N/A"
    )

    lateness_text = (
        f"{average_lateness:.2f} days"
        if pd.notna(average_lateness)
        else "N/A"
    )

    review_text = (
        f"{average_review:.2f}/5"
        if pd.notna(average_review)
        else "N/A"
    )

    repeat_text = (
        f"{repeat_rate:.2f}%"
        if pd.notna(repeat_rate)
        else "N/A"
    )

    return (
        orders_text,
        customers_text,
        late_text,
        lateness_text,
        review_text,
        repeat_text,
        city_delay_figure,
        review_figure
    )


# ============================================================
# 4. DATA-QUALITY CALLBACK
# ============================================================

@app.callback(
    Output("quality-rows", "children"),
    Output("quality-columns", "children"),
    Output("quality-missing", "children"),
    Output("quality-missing-percent", "children"),
    Output("quality-duplicates", "children"),
    Output("missing-values-graph", "figure"),
    Output("quality-table", "data"),
    Output("quality-table", "columns"),

    Input("dataset-selector", "value")
)
def update_data_quality(selected_dataset):

    selected_df = quality_datasets[
        selected_dataset
    ]

    rows = selected_df.shape[0]
    columns = selected_df.shape[1]

    total_cells = rows * columns

    missing_values = (
        selected_df.isnull().sum().sum()
    )

    missing_percentage = (
        missing_values / total_cells * 100
        if total_cells > 0
        else 0
    )

    duplicate_rows = (
        selected_df.duplicated().sum()
    )

    column_quality = pd.DataFrame({
        "column": selected_df.columns,
        "data_type": (
            selected_df.dtypes.astype(str).values
        ),
        "missing_values": (
            selected_df.isnull().sum().values
        ),
        "unique_values": [
            selected_df[column].nunique(
                dropna=True
            )
            for column in selected_df.columns
        ]
    })

    column_quality["missing_percentage"] = (
        column_quality["missing_values"] /
        rows * 100
    ).round(2)

    column_quality = column_quality.sort_values(
        "missing_values",
        ascending=False
    )

    missing_figure = px.bar(
        column_quality,
        x="column",
        y="missing_percentage",
        color="missing_percentage",
        labels={
            "column": "Column",
            "missing_percentage": "Missing values (%)"
        },
        title=(
            f"Missing Values by Column — "
            f"{selected_dataset}"
        ),
        template="simple_white",
        color_continuous_scale="Reds"
    )

    missing_figure.update_layout(
        xaxis_tickangle=-35
    )

    table_columns = [
        {
            "name": "Column",
            "id": "column"
        },
        {
            "name": "Data type",
            "id": "data_type"
        },
        {
            "name": "Missing values",
            "id": "missing_values"
        },
        {
            "name": "Missing %",
            "id": "missing_percentage"
        },
        {
            "name": "Unique values",
            "id": "unique_values"
        }
    ]

    return (
        f"{rows:,}",
        f"{columns:,}",
        f"{missing_values:,}",
        f"{missing_percentage:.2f}%",
        f"{duplicate_rows:,}",
        missing_figure,
        column_quality.to_dict("records"),
        table_columns
    )

In [0]:
print("Starting Olist dashboard...")

app.run(
    debug=False,
    use_reloader=False
)

In [0]:
# ============================================================
# BUSINESS INSIGHTS: DELIVERY DELAYS ANALYSIS
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
import pandas as pd

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 10)

# ============================================================
# 1. KPI OVERVIEW - KEY METRICS
# ============================================================

print("=" * 60)
print("KPI OVERVIEW: DELIVERY DELAYS & CUSTOMER IMPACT")
print("=" * 60)

# Calculate overall delay rate
total_orders = len(analysis_df)
late_orders = analysis_df['late_order'].sum()
delay_rate = (late_orders / total_orders) * 100

print(f"\n📦 TOTAL ORDERS: {total_orders:,}")
print(f"⏰ LATE ORDERS: {late_orders:,}")
print(f"📊 DELAY RATE: {delay_rate:.2f}%")

# Calculate average delay in days (for late orders only)
average_delay_days = analysis_df[analysis_df['late_order'] == 1]['delay_days'].mean()
print(f"⏱️  AVERAGE DELAY: {average_delay_days:.2f} days")

# Split metrics by on-time vs late
on_time_data = analysis_df[analysis_df['late_order'] == 0]
late_data = analysis_df[analysis_df['late_order'] == 1]

print("\n" + "=" * 60)
print("COMPARISON: ON-TIME vs LATE DELIVERIES")
print("=" * 60)

# Review scores
on_time_review = on_time_data['review_score'].mean()
late_review = late_data['review_score'].mean()

print(f"\n⭐ REVIEW SCORE (On-time): {on_time_review:.2f}")
print(f"⭐ REVIEW SCORE (Late): {late_review:.2f}")
print(f"   Difference: {on_time_review - late_review:.2f} points")

# Repeat purchase rates
on_time_repeat = on_time_data['repeat_customer'].mean() * 100
late_repeat = late_data['repeat_customer'].mean() * 100

print(f"\n🔄 REPEAT PURCHASE RATE (On-time): {on_time_repeat:.2f}%")
print(f"🔄 REPEAT PURCHASE RATE (Late): {late_repeat:.2f}%")
print(f"   Difference: {on_time_repeat - late_repeat:.2f} percentage points")


# ============================================================
# 2. VISUALIZATION - KPI OVERVIEW DASHBOARD
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Delivery Delays: Business Impact Dashboard', fontsize=20, fontweight='bold', y=0.995)

# Chart 1: Overall Delay Rate
ax1 = axes[0, 0]
delay_counts = analysis_df['late_order'].value_counts()
colors_delay = ['#2ecc71', '#e74c3c']
ax1.pie(delay_counts, labels=['On-time', 'Late'], autopct='%1.1f%%', 
        colors=colors_delay, startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
ax1.set_title('Overall Delivery Performance', fontsize=14, fontweight='bold', pad=10)

# Chart 2: Average Delay Distribution
ax2 = axes[0, 1]
ax2.hist(late_data['delay_days'], bins=30, color='#e74c3c', alpha=0.7, edgecolor='black')
ax2.axvline(average_delay_days, color='darkred', linestyle='--', linewidth=2, 
            label=f'Mean: {average_delay_days:.1f} days')
ax2.set_xlabel('Delay (days)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax2.set_title('Distribution of Delays (Late Orders Only)', fontsize=14, fontweight='bold', pad=10)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Chart 3: Review Score Comparison
ax3 = axes[0, 2]
review_comparison = pd.DataFrame({
    'Delivery Status': ['On-time', 'Late'],
    'Review Score': [on_time_review, late_review]
})
colors_review = ['#2ecc71', '#e74c3c']
bars = ax3.bar(review_comparison['Delivery Status'], review_comparison['Review Score'], 
               color=colors_review, alpha=0.8, edgecolor='black', linewidth=1.5)
ax3.set_ylabel('Average Review Score', fontsize=11, fontweight='bold')
ax3.set_title('Customer Satisfaction: On-time vs Late', fontsize=14, fontweight='bold', pad=10)
ax3.set_ylim([0, 5])
for bar in bars:
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

# Chart 4: Repeat Purchase Rate Comparison
ax4 = axes[1, 0]
repeat_comparison = pd.DataFrame({
    'Delivery Status': ['On-time', 'Late'],
    'Repeat Rate (%)': [on_time_repeat, late_repeat]
})
bars = ax4.bar(repeat_comparison['Delivery Status'], repeat_comparison['Repeat Rate (%)'], 
               color=colors_review, alpha=0.8, edgecolor='black', linewidth=1.5)
ax4.set_ylabel('Repeat Purchase Rate (%)', fontsize=11, fontweight='bold')
ax4.set_title('Customer Retention: On-time vs Late', fontsize=14, fontweight='bold', pad=10)
for bar in bars:
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

# Chart 5: Review Score Distribution by Delivery Status
ax5 = axes[1, 1]
review_data = [
    on_time_data['review_score'].values,
    late_data['review_score'].values
]
box_plot = ax5.boxplot(review_data, labels=['On-time', 'Late'], 
                        patch_artist=True, widths=0.6)
for patch, color in zip(box_plot['boxes'], colors_review):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax5.set_ylabel('Review Score', fontsize=11, fontweight='bold')
ax5.set_title('Review Score Distribution', fontsize=14, fontweight='bold', pad=10)
ax5.grid(True, alpha=0.3, axis='y')

# Chart 6: Delay Impact Summary (Text Summary)
ax6 = axes[1, 2]
ax6.axis('off')
summary_text = f"""
KEY FINDINGS
{'─' * 40}

📊 Delay Rate: {delay_rate:.1f}%
   ({late_orders:,} out of {total_orders:,} orders)

⏱️  Average Delay: {average_delay_days:.1f} days

⭐ Review Score Impact:
   • On-time: {on_time_review:.2f} ⭐
   • Late: {late_review:.2f} ⭐
   • Loss: {on_time_review - late_review:.2f} points ({((on_time_review - late_review)/on_time_review * 100):.1f}%)

🔄 Repeat Purchase Impact:
   • On-time: {on_time_repeat:.1f}%
   • Late: {late_repeat:.1f}%
   • Loss: {on_time_repeat - late_repeat:.1f} pp
"""
ax6.text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
         verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.tight_layout()
plt.show()


# ============================================================
# 3. STATISTICAL ANALYSIS - RELATIONSHIP TESTS
# ============================================================

print("\n\n" + "=" * 60)
print("STATISTICAL ANALYSIS: IMPACT OF DELAYS")
print("=" * 60)

# Test 1: Delay vs Review Score (Independent t-test)
print("\n📊 TEST 1: Delay Impact on Customer Satisfaction (Review Score)")
print("─" * 60)

# FIX: some orders have no matching review (review_score is NaN after the
# left-merge in the prep cell). stats.ttest_ind propagates NaN by default,
# which is why t_stat/p_value came back as NaN. Drop the missing values first.
on_time_reviews = on_time_data['review_score'].dropna()
late_reviews = late_data['review_score'].dropna()

# Welch's t-test (equal_var=False) is safer here since the two groups have
# very different sizes (n=89,120 vs n=7,864) and likely different variances.
t_stat_review, p_value_review = stats.ttest_ind(
    on_time_reviews,
    late_reviews,
    equal_var=False
)

print(f"\nHypothesis: Late deliveries result in lower review scores")
print(f"\nOn-time Review Score: {on_time_review:.3f} (n={len(on_time_reviews):,})")
print(f"Late Review Score: {late_review:.3f} (n={len(late_reviews):,})")
print(f"Difference: {on_time_review - late_review:.3f} points")
print(f"\nt-statistic: {t_stat_review:.4f}")
print(f"p-value: {p_value_review:.6f}")

if p_value_review < 0.001:
    print(f"\n✅ RESULT: HIGHLY SIGNIFICANT (p < 0.001)")
    print(f"   Late deliveries have a statistically significant negative impact")
    print(f"   on customer satisfaction (review scores).")
elif p_value_review < 0.05:
    print(f"\n✅ RESULT: SIGNIFICANT (p < 0.05)")
    print(f"   Late deliveries significantly reduce customer satisfaction.")
else:
    print(f"\n❌ RESULT: NOT SIGNIFICANT (p >= 0.05)")
    print(f"   No statistically significant difference found.")

# Calculate effect size (Cohen's d)
# FIX: use the same dropna'd n's used in the t-test above, not the raw group
# sizes (which include rows with a missing review_score).
cohens_d_review = (on_time_review - late_review) / np.sqrt(
    ((len(on_time_reviews) - 1) * on_time_reviews.std()**2 + 
     (len(late_reviews) - 1) * late_reviews.std()**2) / 
    (len(on_time_reviews) + len(late_reviews) - 2)
)
print(f"\nEffect Size (Cohen's d): {cohens_d_review:.4f}")
if abs(cohens_d_review) < 0.2:
    effect_label = "Small"
elif abs(cohens_d_review) < 0.5:
    effect_label = "Medium"
else:
    effect_label = "Large"
print(f"Effect Magnitude: {effect_label}")

# Test 2: Delay vs Repeat Purchase (Chi-square test)
print("\n\n📊 TEST 2: Delay Impact on Customer Retention (Repeat Purchase)")
print("─" * 60)

# Create contingency table
contingency_table = pd.crosstab(
    analysis_df['late_order'],
    analysis_df['repeat_customer']
)

print("\nContingency Table:")
print(contingency_table)

chi2, p_value_repeat, dof, expected = stats.chi2_contingency(contingency_table)

print(f"\nHypothesis: Late deliveries reduce repeat purchase likelihood")
print(f"\nOn-time Repeat Rate: {on_time_repeat:.2f}%")
print(f"Late Repeat Rate: {late_repeat:.2f}%")
print(f"Difference: {on_time_repeat - late_repeat:.2f} percentage points")
print(f"\nChi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value_repeat:.6f}")

if p_value_repeat < 0.001:
    print(f"\n✅ RESULT: HIGHLY SIGNIFICANT (p < 0.001)")
    print(f"   Late deliveries have a statistically significant negative impact")
    print(f"   on repeat purchase behavior.")
elif p_value_repeat < 0.05:
    print(f"\n✅ RESULT: SIGNIFICANT (p < 0.05)")
    print(f"   Late deliveries significantly reduce repeat purchase likelihood.")
else:
    print(f"\n❌ RESULT: NOT SIGNIFICANT (p >= 0.05)")
    print(f"   No statistically significant association found.")

# Calculate Cramér's V (effect size for chi-square)
n = len(analysis_df)
cramers_v = np.sqrt(chi2 / (n * (min(contingency_table.shape) - 1)))
print(f"\nEffect Size (Cramér's V): {cramers_v:.4f}")
if cramers_v < 0.1:
    effect_label_v = "Small"
elif cramers_v < 0.3:
    effect_label_v = "Medium"
else:
    effect_label_v = "Large"
print(f"Effect Magnitude: {effect_label_v}")


# ============================================================
# 4. ADVANCED VISUALIZATIONS - RELATIONSHIPS
# ============================================================

fig2, axes2 = plt.subplots(1, 2, figsize=(16, 6))
fig2.suptitle('Statistical Relationships: Delays, Satisfaction & Retention', 
              fontsize=18, fontweight='bold')

# Chart 1: Scatter plot - Delay days vs Review Score
ax_scatter1 = axes2[0]
ax_scatter1.scatter(late_data['delay_days'], late_data['review_score'], 
                    alpha=0.5, s=30, color='#e74c3c', edgecolor='black', linewidth=0.5)
z = np.polyfit(late_data['delay_days'], late_data['review_score'], 1)
p = np.poly1d(z)
ax_scatter1.plot(late_data['delay_days'], p(late_data['delay_days']), 
                 "--", color='darkred', linewidth=2, label=f'Trend line')
corr_delay_review = late_data[['delay_days', 'review_score']].corr().iloc[0, 1]
ax_scatter1.set_xlabel('Delay (days)', fontsize=12, fontweight='bold')
ax_scatter1.set_ylabel('Review Score', fontsize=12, fontweight='bold')
ax_scatter1.set_title(f'Delay Duration vs Customer Satisfaction\n(Correlation: {corr_delay_review:.3f})', 
                      fontsize=13, fontweight='bold', pad=10)
ax_scatter1.legend(fontsize=10)
ax_scatter1.grid(True, alpha=0.3)

# Chart 2: Grouped bar chart - Repeat purchase by delay severity
ax_bar = axes2[1]
analysis_df_temp = analysis_df.copy()
analysis_df_temp['delay_category'] = pd.cut(
    analysis_df_temp['delay_days'],
    bins=[-np.inf, 0, 3, 7, np.inf],
    labels=['On-time', '1-3 days late', '4-7 days late', '7+ days late']
)
repeat_by_delay = analysis_df_temp.groupby('delay_category')['repeat_customer'].agg(['mean', 'count'])
repeat_by_delay['mean'] = repeat_by_delay['mean'] * 100

colors_gradient = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
bars = ax_bar.bar(range(len(repeat_by_delay)), repeat_by_delay['mean'], 
                   color=colors_gradient, alpha=0.8, edgecolor='black', linewidth=1.5)
ax_bar.set_xticks(range(len(repeat_by_delay)))
ax_bar.set_xticklabels(repeat_by_delay.index, fontsize=10, fontweight='bold')
ax_bar.set_ylabel('Repeat Purchase Rate (%)', fontsize=12, fontweight='bold')
ax_bar.set_title('Customer Retention by Delay Severity', fontsize=13, fontweight='bold', pad=10)
for i, (bar, count) in enumerate(zip(bars, repeat_by_delay['count'])):
    height = bar.get_height()
    ax_bar.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.1f}%\n(n={count:,})', 
               ha='center', va='bottom', fontsize=9, fontweight='bold')
ax_bar.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


# ============================================================
# 5. BUSINESS IMPACT SUMMARY
# ============================================================

print("\n\n" + "=" * 60)
print("BUSINESS IMPACT SUMMARY")
print("=" * 60)

print(f"""
🎯 EXECUTIVE SUMMARY
{"─" * 60}

1️⃣  DELAY PERFORMANCE:
   • {delay_rate:.1f}% of orders arrive late
   • Average delay duration: {average_delay_days:.1f} days
   • Total late orders: {late_orders:,}

2️⃣  CUSTOMER SATISFACTION IMPACT:
   • Review score drops by {on_time_review - late_review:.2f} points for late deliveries
   • Statistical significance: p = {p_value_review:.6f} (***)
   • Effect size: {effect_label} (Cohen's d = {cohens_d_review:.3f})
   • ✅ PROVEN: Delays significantly harm customer satisfaction

3️⃣  CUSTOMER RETENTION IMPACT:
   • Repeat purchase rate drops by {on_time_repeat - late_repeat:.1f} percentage points
   • Statistical significance: p = {p_value_repeat:.6f} (***)
   • Effect size: {effect_label_v} (Cramér's V = {cramers_v:.3f})
   • ✅ PROVEN: Delays significantly reduce repeat purchases

4️⃣  FINANCIAL IMPLICATIONS:
   • Estimated lost repeat customers: {int((on_time_repeat - late_repeat) / 100 * late_orders):,}
   • Severity correlation: Longer delays → worse retention
   • Orders delayed 7+ days show the steepest retention drop

💡 RECOMMENDATION:
   Prioritize reducing delivery delays to improve both customer
   satisfaction and long-term retention. Focus especially on 
   preventing delays beyond 7 days, which show the worst impact.
""")

print("=" * 60)

In [0]:


import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from dash import (
    Dash,
    dcc,
    html,
    Input,
    Output,
    dash_table
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix
)


# ============================================================
# 1. LOAD ORIGINAL DATASETS
# Used by the Data Quality tab
# ============================================================

original_customers_df = load_file(
    "olist_customers_dataset.csv"
)

original_geolocation_df = load_file(
    "olist_geolocation_dataset.csv"
)

original_orders_df = load_file(
    "olist_orders_dataset.csv"
)

original_order_items_df = load_file(
    "olist_order_items_dataset.csv"
)

original_order_payments_df = load_file(
    "olist_order_payments_dataset.csv"
)

original_order_reviews_df = load_file(
    "olist_order_reviews_dataset.csv"
)

original_products_df = load_file(
    "olist_products_dataset.csv"
)

original_sellers_df = load_file(
    "olist_sellers_dataset.csv"
)

original_category_translation_df = load_file(
    "product_category_name_translation.csv"
)


quality_datasets = {
    "Customers": original_customers_df,
    "Geolocation": original_geolocation_df,
    "Orders": original_orders_df,
    "Order Items": original_order_items_df,
    "Order Payments": original_order_payments_df,
    "Order Reviews": original_order_reviews_df,
    "Products": original_products_df,
    "Sellers": original_sellers_df,
    "Product Category Translation":
        original_category_translation_df
}


# ============================================================
# 2. PREPARE MODEL RESULTS
# ============================================================

def create_model_result(
    model,
    feature_set,
    y_true,
    predictions
):
    return {
        "model": model,
        "feature_set": feature_set,
        "accuracy": accuracy_score(
            y_true,
            predictions
        ),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        )
    }


model_results_df = pd.DataFrame([
    create_model_result(
        "Logistic Regression",
        "Original",
        Y_test,
        delay_predictions
    ),

    create_model_result(
        "Logistic Regression",
        "Transformed",
        Y_test_log,
        log_predictions
    ),

    create_model_result(
        "Random Forest",
        "Original",
        Y_test,
        rf_predictions
    ),

    create_model_result(
        "Random Forest",
        "Transformed",
        Y_test_rf_log,
        transformed_rf_predictions
    ),

    create_model_result(
        "Neural Network",
        "Original",
        Y_test,
        nn_predictions
    )
])


model_results_df["model_label"] = (
    model_results_df["model"] +
    " - " +
    model_results_df["feature_set"]
)


model_confusion_matrices = {
    ("Logistic Regression", "Original"):
        confusion_matrix(
            Y_test,
            delay_predictions
        ),

    ("Logistic Regression", "Transformed"):
        confusion_matrix(
            Y_test_log,
            log_predictions
        ),

    ("Random Forest", "Original"):
        confusion_matrix(
            Y_test,
            rf_predictions
        ),

    ("Random Forest", "Transformed"):
        confusion_matrix(
            Y_test_rf_log,
            transformed_rf_predictions
        ),

    ("Neural Network", "Original"):
        confusion_matrix(
            Y_test,
            nn_predictions
        )
}


# Data for model-comparison chart

model_results_long = model_results_df.melt(
    id_vars=[
        "model",
        "feature_set",
        "model_label"
    ],
    value_vars=[
        "accuracy",
        "precision",
        "recall"
    ],
    var_name="metric",
    value_name="score"
)

model_comparison_figure = px.bar(
    model_results_long,
    x="model_label",
    y="score",
    color="metric",
    barmode="group",
    labels={
        "model_label": "Model",
        "score": "Score",
        "metric": "Metric"
    },
    title="Comparison of All Delay-Prediction Models",
    template="simple_white",
    color_discrete_map={
        "accuracy": "#2563EB",
        "precision": "#F59E0B",
        "recall": "#10B981"
    }
)

model_comparison_figure.update_yaxes(
    range=[0, 1],
    tickformat=".0%"
)

model_comparison_figure.update_layout(
    xaxis_tickangle=-20,
    legend_title_text=""
)


# ============================================================
# 3. PREPARE BUSINESS KPI DATA
# ============================================================

# Remove dashboard columns if this cell was run previously

columns_to_remove = [
    column
    for column in analysis_df.columns
    if column.startswith(
        (
            "customer_unique_id",
            "customer_city",
            "review_score",
            "repeat_purchase",
            "next_purchase_date"
        )
    )
]

business_df = (
    analysis_df
    .drop(
        columns=columns_to_remove,
        errors="ignore"
    )
    .drop_duplicates("order_id")
    .copy()
)


# Add permanent customer ID and city

customer_information = (
    original_customers_df[
        [
            "customer_id",
            "customer_unique_id",
            "customer_city"
        ]
    ]
    .drop_duplicates("customer_id")
)

business_df = business_df.merge(
    customer_information,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

business_df["customer_city"] = (
    business_df["customer_city"]
    .str.strip()
    .str.title()
)


# Add average review score per order

reviews_per_order = (
    original_order_reviews_df
    .groupby(
        "order_id",
        as_index=False
    )
    .agg(
        review_score=("review_score", "mean")
    )
)

business_df = business_df.merge(
    reviews_per_order,
    on="order_id",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 4. ENGINEER REPEAT-PURCHASE FEATURE
# ============================================================

order_history = original_orders_df[
    [
        "order_id",
        "customer_id",
        "order_purchase_timestamp",
        "order_status"
    ]
].copy()

order_history["order_purchase_timestamp"] = pd.to_datetime(
    order_history["order_purchase_timestamp"],
    errors="coerce"
)

# Exclude unsuccessful purchases

order_history = order_history[
    ~order_history["order_status"].isin(
        ["canceled", "unavailable"]
    )
].copy()

order_history = order_history.merge(
    original_customers_df[
        [
            "customer_id",
            "customer_unique_id"
        ]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# Sort each customer's purchases chronologically

order_history = order_history.sort_values(
    [
        "customer_unique_id",
        "order_purchase_timestamp"
    ]
)

# Find the next purchase made by the same customer

order_history["next_purchase_date"] = (
    order_history
    .groupby("customer_unique_id")[
        "order_purchase_timestamp"
    ]
    .shift(-1)
)

order_history["repeat_purchase"] = (
    order_history["next_purchase_date"].notna()
).astype(int)

business_df = business_df.merge(
    order_history[
        [
            "order_id",
            "repeat_purchase"
        ]
    ],
    on="order_id",
    how="left",
    validate="one_to_one"
)

business_df["repeat_purchase"] = (
    business_df["repeat_purchase"]
    .fillna(0)
    .astype(int)
)


# Standardise dashboard variables

business_df["delivery_status"] = np.where(
    business_df["delay_days"] > 0,
    "Delayed",
    "On time or early"
)

business_df["order_purchase_timestamp"] = pd.to_datetime(
    business_df["order_purchase_timestamp"],
    errors="coerce"
)


# ============================================================
# 5. DASHBOARD FILTER VALUES
# ============================================================

state_options = [
    {
        "label": "All states",
        "value": "All"
    }
] + [
    {
        "label": state,
        "value": state
    }
    for state in sorted(
        business_df["customer_state"]
        .dropna()
        .unique()
    )
]

minimum_date = (
    business_df["order_purchase_timestamp"]
    .dropna()
    .min()
    .date()
)

maximum_date = (
    business_df["order_purchase_timestamp"]
    .dropna()
    .max()
    .date()
)


# ============================================================
# 6. DASHBOARD HELPER FUNCTIONS
# ============================================================

def create_kpi_card(
    title,
    component_id,
    colour="#2563EB"
):
    return html.Div(
        [
            html.P(
                title,
                style={
                    "color": "#64748B",
                    "margin": "0",
                    "fontSize": "14px"
                }
            ),

            html.P(
                id=component_id,
                children="Loading...",
                style={
                    "fontSize": "25px",
                    "fontWeight": "bold",
                    "color": "#1E3A5F",
                    "margin": "8px 0 0 0"
                }
            )
        ],
        style={
            "backgroundColor": "#F8FAFC",
            "padding": "17px",
            "borderRadius": "10px",
            "borderLeft": f"5px solid {colour}",
            "textAlign": "center"
        }
    )


def create_empty_figure(message):

    figure = go.Figure()

    figure.add_annotation(
        text=message,
        x=0.5,
        y=0.5,
        xref="paper",
        yref="paper",
        showarrow=False,
        font={
            "size": 17,
            "color": "#64748B"
        }
    )

    figure.update_layout(
        template="simple_white",
        xaxis={"visible": False},
        yaxis={"visible": False}
    )

    return figure


# ============================================================
# 7. DASHBOARD STYLES
# ============================================================

PAGE_STYLE = {
    "fontFamily": "Arial, sans-serif",
    "backgroundColor": "#F3F6FA",
    "minHeight": "100vh",
    "padding": "25px"
}

SECTION_STYLE = {
    "backgroundColor": "white",
    "padding": "22px",
    "marginTop": "18px",
    "borderRadius": "12px",
    "boxShadow": "0 2px 8px rgba(0,0,0,0.08)"
}

FILTER_GRID_STYLE = {
    "display": "grid",
    "gridTemplateColumns":
        "repeat(auto-fit, minmax(190px, 1fr))",
    "gap": "15px",
    "marginBottom": "20px"
}

KPI_GRID_STYLE = {
    "display": "grid",
    "gridTemplateColumns":
        "repeat(auto-fit, minmax(160px, 1fr))",
    "gap": "15px",
    "marginBottom": "20px"
}

GRAPH_GRID_STYLE = {
    "display": "grid",
    "gridTemplateColumns":
        "repeat(auto-fit, minmax(400px, 1fr))",
    "gap": "18px"
}


# ============================================================
# 8. CREATE DASH APPLICATION
# ============================================================

app = Dash(__name__)


# ============================================================
# 9. DASHBOARD LAYOUT
# ============================================================

app.layout = html.Div(
    [
        # Header

        html.Div(
            [
                html.H1(
                    "Olist Delivery Performance Dashboard",
                    style={
                        "margin": "0 0 6px 0",
                        "color": "#1E3A5F"
                    }
                ),

                html.P(
                    "Delay prediction, regional business "
                    "performance and original dataset quality",
                    style={
                        "color": "#64748B",
                        "fontSize": "17px",
                        "margin": "0"
                    }
                )
            ],
            style={
                "backgroundColor": "white",
                "padding": "22px",
                "borderRadius": "12px",
                "borderTop": "6px solid #2563EB"
            }
        ),

        dcc.Tabs(
            [
                # =================================================
                # MODEL TAB
                # =================================================

                dcc.Tab(
                    label="Model Selection",
                    children=[
                        html.Div(
                            [
                                html.H2(
                                    "Delay-Prediction Models"
                                ),

                                html.P(
                                    "Select a model and the feature "
                                    "engineering method."
                                ),

                                html.Div(
                                    [
                                        html.Div(
                                            [
                                                html.Label("Model"),

                                                dcc.Dropdown(
                                                    id="model-selector",
                                                    options=[
                                                        {
                                                            "label":
                                                                "Logistic Regression",
                                                            "value":
                                                                "Logistic Regression"
                                                        },
                                                        {
                                                            "label":
                                                                "Random Forest",
                                                            "value":
                                                                "Random Forest"
                                                        },
                                                        {
                                                            "label":
                                                                "Neural Network",
                                                            "value":
                                                                "Neural Network"
                                                        }
                                                    ],
                                                    value="Random Forest",
                                                    clearable=False
                                                )
                                            ]
                                        ),

                                        html.Div(
                                            [
                                                html.Label(
                                                    "Feature set"
                                                ),

                                                dcc.Dropdown(
                                                    id="transformation-selector",
                                                    options=[
                                                        {
                                                            "label":
                                                                "Original features",
                                                            "value":
                                                                "Original"
                                                        },
                                                        {
                                                            "label":
                                                                "Transformed features",
                                                            "value":
                                                                "Transformed"
                                                        }
                                                    ],
                                                    value="Original",
                                                    clearable=False
                                                )
                                            ]
                                        )
                                    ],
                                    style=FILTER_GRID_STYLE
                                ),

                                html.H3(
                                    id="selected-model-title",
                                    children="Selected model",
                                    style={
                                        "color": "#1E3A5F"
                                    }
                                ),

                                html.Div(
                                    [
                                        create_kpi_card(
                                            "Accuracy",
                                            "model-accuracy"
                                        ),

                                        create_kpi_card(
                                            "Precision",
                                            "model-precision",
                                            "#F59E0B"
                                        ),

                                        create_kpi_card(
                                            "Recall",
                                            "model-recall",
                                            "#10B981"
                                        )
                                    ],
                                    style=KPI_GRID_STYLE
                                ),

                                html.Div(
                                    [
                                        dcc.Graph(
                                            id="confusion-matrix-graph"
                                        ),

                                        dcc.Graph(
                                            figure=model_comparison_figure
                                        )
                                    ],
                                    style=GRAPH_GRID_STYLE
                                )
                            ],
                            style=SECTION_STYLE
                        )
                    ]
                ),

                # =================================================
                # BUSINESS KPI TAB
                # =================================================

                dcc.Tab(
                    label="Business KPIs",
                    children=[
                        html.Div(
                            [
                                html.H2(
                                    "Regional Delivery Performance"
                                ),

                                html.P(
                                    "Filter by location, delivery status "
                                    "and purchase date."
                                ),

                                html.Div(
                                    [
                                        html.Div(
                                            [
                                                html.Label("State"),

                                                dcc.Dropdown(
                                                    id="state-filter",
                                                    options=state_options,
                                                    value="All",
                                                    clearable=False
                                                )
                                            ]
                                        ),

                                        html.Div(
                                            [
                                                html.Label("City"),

                                                dcc.Dropdown(
                                                    id="city-filter",
                                                    options=[
                                                        {
                                                            "label":
                                                                "All cities",
                                                            "value":
                                                                "All"
                                                        }
                                                    ],
                                                    value="All",
                                                    clearable=False
                                                )
                                            ]
                                        ),

                                        html.Div(
                                            [
                                                html.Label(
                                                    "Delivery status"
                                                ),

                                                dcc.Dropdown(
                                                    id="delivery-filter",
                                                    options=[
                                                        {
                                                            "label":
                                                                "All orders",
                                                            "value":
                                                                "All"
                                                        },
                                                        {
                                                            "label":
                                                                "On time or early",
                                                            "value":
                                                                "On time or early"
                                                        },
                                                        {
                                                            "label":
                                                                "Delayed",
                                                            "value":
                                                                "Delayed"
                                                        }
                                                    ],
                                                    value="All",
                                                    clearable=False
                                                )
                                            ]
                                        ),

                                        html.Div(
                                            [
                                                html.Label(
                                                    "Purchase dates"
                                                ),

                                                dcc.DatePickerRange(
                                                    id="date-filter",
                                                    min_date_allowed=
                                                        minimum_date,
                                                    max_date_allowed=
                                                        maximum_date,
                                                    start_date=
                                                        minimum_date,
                                                    end_date=
                                                        maximum_date
                                                )
                                            ]
                                        )
                                    ],
                                    style=FILTER_GRID_STYLE
                                ),

                                html.Div(
                                    [
                                        create_kpi_card(
                                            "Delivered orders",
                                            "kpi-orders"
                                        ),

                                        create_kpi_card(
                                            "Unique customers",
                                            "kpi-customers"
                                        ),

                                        create_kpi_card(
                                            "Late-order rate",
                                            "kpi-late-rate",
                                            "#EF4444"
                                        ),

                                        create_kpi_card(
                                            "Average lateness",
                                            "kpi-average-late",
                                            "#F59E0B"
                                        ),

                                        create_kpi_card(
                                            "Average review",
                                            "kpi-review",
                                            "#8B5CF6"
                                        ),

                                        create_kpi_card(
                                            "Repeat-purchase rate",
                                            "kpi-repeat",
                                            "#10B981"
                                        )
                                    ],
                                    style=KPI_GRID_STYLE
                                ),

                                html.Div(
                                    [
                                        dcc.Graph(
                                            id="city-delay-graph"
                                        ),

                                        dcc.Graph(
                                            id="review-status-graph"
                                        )
                                    ],
                                    style=GRAPH_GRID_STYLE
                                )
                            ],
                            style=SECTION_STYLE
                        )
                    ]
                ),

                # =================================================
                # DATA QUALITY TAB
                # =================================================

                dcc.Tab(
                    label="Data Quality",
                    children=[
                        html.Div(
                            [
                                html.H2(
                                    "Original Dataset Quality"
                                ),

                                html.P(
                                    "The results in this tab come from "
                                    "the original CSV files before cleaning."
                                ),

                                html.Div(
                                    [
                                        html.Div(
                                            [
                                                html.Label(
                                                    "Dataset"
                                                ),

                                                dcc.Dropdown(
                                                    id="dataset-selector",
                                                    options=[
                                                        {
                                                            "label": name,
                                                            "value": name
                                                        }
                                                        for name in
                                                        quality_datasets.keys()
                                                    ],
                                                    value="Orders",
                                                    clearable=False
                                                )
                                            ]
                                        )
                                    ],
                                    style=FILTER_GRID_STYLE
                                ),

                                html.Div(
                                    [
                                        create_kpi_card(
                                            "Rows",
                                            "quality-rows"
                                        ),

                                        create_kpi_card(
                                            "Columns",
                                            "quality-columns"
                                        ),

                                        create_kpi_card(
                                            "Missing values",
                                            "quality-missing",
                                            "#F59E0B"
                                        ),

                                        create_kpi_card(
                                            "Missing cells",
                                            "quality-missing-percent",
                                            "#EF4444"
                                        ),

                                        create_kpi_card(
                                            "Duplicate rows",
                                            "quality-duplicates",
                                            "#8B5CF6"
                                        )
                                    ],
                                    style=KPI_GRID_STYLE
                                ),

                                dcc.Graph(
                                    id="missing-values-graph"
                                ),

                                html.H3(
                                    "Column-Level Quality"
                                ),

                                dash_table.DataTable(
                                    id="quality-table",
                                    page_size=12,
                                    sort_action="native",
                                    filter_action="native",

                                    style_table={
                                        "overflowX": "auto"
                                    },

                                    style_header={
                                        "backgroundColor": "#1E3A5F",
                                        "color": "white",
                                        "fontWeight": "bold"
                                    },

                                    style_cell={
                                        "padding": "9px",
                                        "textAlign": "left",
                                        "fontFamily": "Arial",
                                        "fontSize": "13px"
                                    },

                                    style_data_conditional=[
                                        {
                                            "if": {
                                                "filter_query":
                                                    "{missing_values} > 0"
                                            },
                                            "backgroundColor": "#FEF3C7"
                                        }
                                    ]
                                )
                            ],
                            style=SECTION_STYLE
                        )
                    ]
                )
            ],
            style={
                "marginTop": "18px"
            }
        )
    ],
    style=PAGE_STYLE
)


# ============================================================
# 10. MODEL CALLBACKS
# ============================================================

@app.callback(
    Output(
        "transformation-selector",
        "options"
    ),
    Output(
        "transformation-selector",
        "value"
    ),
    Input(
        "model-selector",
        "value"
    )
)
def update_transformation_options(selected_model):

    if selected_model == "Neural Network":

        options = [
            {
                "label": "Original features",
                "value": "Original"
            }
        ]

    else:

        options = [
            {
                "label": "Original features",
                "value": "Original"
            },
            {
                "label": "Transformed features",
                "value": "Transformed"
            }
        ]

    return options, "Original"


@app.callback(
    Output(
        "selected-model-title",
        "children"
    ),
    Output(
        "model-accuracy",
        "children"
    ),
    Output(
        "model-precision",
        "children"
    ),
    Output(
        "model-recall",
        "children"
    ),
    Output(
        "confusion-matrix-graph",
        "figure"
    ),

    Input(
        "model-selector",
        "value"
    ),
    Input(
        "transformation-selector",
        "value"
    )
)
def update_model_results(
    selected_model,
    selected_feature_set
):

    if selected_feature_set is None:
        selected_feature_set = "Original"

    selected_result = model_results_df[
        (model_results_df["model"] == selected_model) &
        (
            model_results_df["feature_set"] ==
            selected_feature_set
        )
    ]

    if selected_result.empty:

        selected_feature_set = "Original"

        selected_result = model_results_df[
            (model_results_df["model"] == selected_model) &
            (
                model_results_df["feature_set"] ==
                selected_feature_set
            )
        ]

    selected_result = selected_result.iloc[0]

    confusion = model_confusion_matrices[
        (
            selected_model,
            selected_feature_set
        )
    ]

    confusion_figure = px.imshow(
        confusion,
        text_auto=True,
        color_continuous_scale="Blues",
        x=[
            "Predicted on time",
            "Predicted delayed"
        ],
        y=[
            "Actually on time",
            "Actually delayed"
        ],
        labels={
            "x": "Prediction",
            "y": "Actual result",
            "color": "Orders"
        },
        title="Confusion Matrix"
    )

    confusion_figure.update_layout(
        template="simple_white"
    )

    title = (
        f"{selected_model} — "
        f"{selected_feature_set} features"
    )

    return (
        title,
        f"{selected_result['accuracy']:.1%}",
        f"{selected_result['precision']:.1%}",
        f"{selected_result['recall']:.1%}",
        confusion_figure
    )


# ============================================================
# 11. CITY FILTER CALLBACK
# ============================================================

@app.callback(
    Output(
        "city-filter",
        "options"
    ),
    Output(
        "city-filter",
        "value"
    ),
    Input(
        "state-filter",
        "value"
    )
)
def update_city_options(selected_state):

    filtered_cities = business_df.copy()

    if selected_state not in [None, "All"]:
        filtered_cities = filtered_cities[
            filtered_cities["customer_state"] ==
            selected_state
        ]

    cities = sorted(
        filtered_cities["customer_city"]
        .dropna()
        .unique()
    )

    city_options = [
        {
            "label": "All cities",
            "value": "All"
        }
    ] + [
        {
            "label": city,
            "value": city
        }
        for city in cities
    ]

    return city_options, "All"


# ============================================================
# 12. BUSINESS KPI CALLBACK
# ============================================================

@app.callback(
    Output("kpi-orders", "children"),
    Output("kpi-customers", "children"),
    Output("kpi-late-rate", "children"),
    Output("kpi-average-late", "children"),
    Output("kpi-review", "children"),
    Output("kpi-repeat", "children"),
    Output("city-delay-graph", "figure"),
    Output("review-status-graph", "figure"),

    Input("state-filter", "value"),
    Input("city-filter", "value"),
    Input("delivery-filter", "value"),
    Input("date-filter", "start_date"),
    Input("date-filter", "end_date")
)
def update_business_kpis(
    selected_state,
    selected_city,
    selected_delivery_status,
    start_date,
    end_date
):

    filtered_df = business_df.copy()

    if selected_state not in [None, "All"]:
        filtered_df = filtered_df[
            filtered_df["customer_state"] ==
            selected_state
        ]

    if selected_city not in [None, "All"]:
        filtered_df = filtered_df[
            filtered_df["customer_city"] ==
            selected_city
        ]

    if selected_delivery_status not in [None, "All"]:
        filtered_df = filtered_df[
            filtered_df["delivery_status"] ==
            selected_delivery_status
        ]

    if start_date is not None:
        filtered_df = filtered_df[
            filtered_df["order_purchase_timestamp"] >=
            pd.to_datetime(start_date)
        ]

    if end_date is not None:
        filtered_df = filtered_df[
            filtered_df["order_purchase_timestamp"] <
            pd.to_datetime(end_date) +
            pd.Timedelta(days=1)
        ]

    # No data found

    if filtered_df.empty:

        empty_figure = create_empty_figure(
            "No orders match the selected filters"
        )

        return (
            "0",
            "0",
            "N/A",
            "N/A",
            "N/A",
            "N/A",
            empty_figure,
            empty_figure
        )

    # KPI calculations

    number_of_orders = (
        filtered_df["order_id"].nunique()
    )

    unique_customers = (
        filtered_df["customer_unique_id"].nunique()
    )

    late_rate = (
        filtered_df["late_order"].mean() * 100
    )

    late_orders = filtered_df[
        filtered_df["delay_days"] > 0
    ]

    average_lateness = (
        late_orders["delay_days"].mean()
    )

    average_review = (
        filtered_df["review_score"].mean()
    )

    repeat_purchase_rate = (
        filtered_df["repeat_purchase"].mean() * 100
    )

    # City chart

    city_summary = (
        filtered_df
        .dropna(subset=["customer_city"])
        .groupby("customer_city")
        .agg(
            order_count=("order_id", "nunique"),
            late_order_rate=("late_order", "mean"),
            average_review=("review_score", "mean")
        )
        .reset_index()
    )

    city_summary["late_order_rate"] = (
        city_summary["late_order_rate"] * 100
    )

    city_summary = (
        city_summary
        .sort_values(
            "order_count",
            ascending=False
        )
        .head(15)
        .sort_values(
            "late_order_rate",
            ascending=False
        )
    )

    if city_summary.empty:

        city_figure = create_empty_figure(
            "No city data available"
        )

    else:

        city_figure = px.bar(
            city_summary,
            x="customer_city",
            y="late_order_rate",
            color="average_review",
            hover_data=["order_count"],
            labels={
                "customer_city": "Customer city",
                "late_order_rate": "Late orders (%)",
                "average_review": "Average review",
                "order_count": "Orders"
            },
            title="Late-Order Rate in the Largest Selected Cities",
            template="simple_white",
            color_continuous_scale="RdYlGn"
        )

        city_figure.update_layout(
            xaxis_tickangle=-30
        )

    # Review chart

    review_summary = (
        filtered_df
        .dropna(subset=["review_score"])
        .groupby(
            "delivery_status",
            as_index=False
        )
        .agg(
            average_review_score=(
                "review_score",
                "mean"
            ),
            number_of_reviews=(
                "review_score",
                "count"
            )
        )
    )

    if review_summary.empty:

        review_figure = create_empty_figure(
            "No review data available"
        )

    else:

        review_figure = px.bar(
            review_summary,
            x="delivery_status",
            y="average_review_score",
            color="delivery_status",
            hover_data=["number_of_reviews"],
            labels={
                "delivery_status": "Delivery status",
                "average_review_score":
                    "Average review score",
                "number_of_reviews":
                    "Reviews"
            },
            title="Average Review Score by Delivery Status",
            template="simple_white",
            color_discrete_map={
                "On time or early": "#10B981",
                "Delayed": "#EF4444"
            }
        )

        review_figure.update_yaxes(
            range=[0, 5]
        )

        review_figure.update_layout(
            showlegend=False
        )

    # Format results

    average_lateness_text = (
        f"{average_lateness:.2f} days"
        if pd.notna(average_lateness)
        else "N/A"
    )

    average_review_text = (
        f"{average_review:.2f}/5"
        if pd.notna(average_review)
        else "N/A"
    )

    return (
        f"{number_of_orders:,}",
        f"{unique_customers:,}",
        f"{late_rate:.2f}%",
        average_lateness_text,
        average_review_text,
        f"{repeat_purchase_rate:.2f}%",
        city_figure,
        review_figure
    )


# ============================================================
# 13. DATA QUALITY CALLBACK
# ============================================================

@app.callback(
    Output("quality-rows", "children"),
    Output("quality-columns", "children"),
    Output("quality-missing", "children"),
    Output("quality-missing-percent", "children"),
    Output("quality-duplicates", "children"),
    Output("missing-values-graph", "figure"),
    Output("quality-table", "data"),
    Output("quality-table", "columns"),

    Input("dataset-selector", "value")
)
def update_data_quality(selected_dataset):

    selected_df = quality_datasets[
        selected_dataset
    ]

    rows = selected_df.shape[0]
    number_of_columns = selected_df.shape[1]

    total_cells = rows * number_of_columns

    missing_values = int(
        selected_df.isnull().sum().sum()
    )

    missing_percentage = (
        missing_values / total_cells * 100
        if total_cells > 0
        else 0
    )

    duplicate_rows = int(
        selected_df.duplicated().sum()
    )

    column_quality = pd.DataFrame({
        "column": selected_df.columns.astype(str),

        "data_type":
            selected_df.dtypes.astype(str).values,

        "missing_values":
            selected_df.isnull().sum().values,

        "unique_values": [
            selected_df[column].nunique(
                dropna=True
            )
            for column in selected_df.columns
        ]
    })

    if rows > 0:

        column_quality["missing_percentage"] = (
            column_quality["missing_values"] /
            rows * 100
        ).round(2)

    else:

        column_quality["missing_percentage"] = 0

    column_quality = column_quality.sort_values(
        "missing_values",
        ascending=False
    )

    missing_figure = px.bar(
        column_quality,
        x="column",
        y="missing_percentage",
        color="missing_percentage",
        labels={
            "column": "Column",
            "missing_percentage": "Missing values (%)"
        },
        title=(
            f"Missing Values by Column — "
            f"{selected_dataset}"
        ),
        template="simple_white",
        color_continuous_scale="Reds"
    )

    missing_figure.update_layout(
        xaxis_tickangle=-35
    )

    table_columns = [
        {
            "name": "Column",
            "id": "column"
        },
        {
            "name": "Data type",
            "id": "data_type"
        },
        {
            "name": "Missing values",
            "id": "missing_values"
        },
        {
            "name": "Missing %",
            "id": "missing_percentage"
        },
        {
            "name": "Unique values",
            "id": "unique_values"
        }
    ]

    return (
        f"{rows:,}",
        f"{number_of_columns:,}",
        f"{missing_values:,}",
        f"{missing_percentage:.2f}%",
        f"{duplicate_rows:,}",
        missing_figure,
        column_quality.to_dict("records"),
        table_columns
    )


# ============================================================
# 14. FINAL CHECKS
# ============================================================

print("Dashboard prepared successfully.")
print("Business observations:", len(business_df))
print("Original quality datasets:", len(quality_datasets))
print("Models available:", len(model_results_df))

In [0]:
print("Starting Olist dashboard...")

app.run(
    debug=False,
    use_reloader=False
)